# Fabric Inventory Explorer

Use this notebook to inspect the complete **collected** inventory. Power BI is optional.
All collector, setup, and report-definition code is embedded. No local deployment scripts are required.
The sharing template opens in `help` mode. A configured deployment opens saved inventory.

## Quick start
1. Upload to a Fabric workspace on an active Fabric capacity.
2. Attach a dedicated, schema-enabled lakehouse as the default. Normally leave all ID overrides blank.
3. If needed, use `MODE = "setup_lakehouse"` to create one. Attach the returned lakehouse and restart the session.
4. Set `MODE = "collect_and_view"` and `CONFIRM_WRITE_TARGET = True` for the first collection.
5. Use `MODE = "view_saved"` to browse. It does not collect, provision, or write data.
6. Optional: after collection, use `MODE = "create_report"` to create the Power BI model and report.

Uploading the notebook alone is enough to obtain the code, not the required workspace, capacity, or permissions.
The first collection needs an active host capacity and appropriate source access.
Other source capacities are **not** resumed by this notebook.
Existing Delta data is preserved by default.

## Contents
- Settings and destination
- Explicit lakehouse setup, collection, and report actions
- Explorer controls: Overview, Capacities, Workspaces, Items
- Deep metadata, Access, Usage, Dependencies
- Changes, Diagnostics, and the complete Collection browser
- Direct view recipes and recovery guidance

**Limits:** permissions describe observed assignments, not complete row-level authorization.
Usage exposes identifiable recorded actors, not necessarily every embedded end user.
The Activity Events API retains 28 days. Manual collection gaps can be permanent.

## Settings

Collection filters above are separate from the viewing filters below.
The `replace_inventory` write mode is destructive. It requires Boolean `CONFIRM_OVERWRITE = True`.
It replaces only this notebook's allowlisted inventory tables in the configured lakehouse.
Do not enable replacement to refresh ordinary inventory; use append snapshots.

For a recipient, destination IDs and names normally come from the attached default lakehouse.
Only advanced users need explicit overrides. The configured copy preserves its verified deployment IDs.
`CONFIRM_WRITE_TARGET` confirms the destination, not permission to overwrite.
Do not point this notebook at a source business lakehouse.

`setup_lakehouse` and `create_report` are explicit authoring actions.
They need destination rights. Neither action grants rights or changes capacity.

In [ ]:
MODE = 'help'
CONFIRM_WRITE_TARGET = False
SETUP_LAKEHOUSE_NAME = "FabricInventory"
REPORT_MODEL_NAME = "Fabric Inventory Model"
REPORT_NAME = "Fabric Inventory"

# Advanced overrides: leave blank in a new recipient's notebook.
TENANT_ID = ''
INVENTORY_WORKSPACE_ID = ''
INVENTORY_WORKSPACE_NAME = ''
INVENTORY_LAKEHOUSE_ID = ''
INVENTORY_LAKEHOUSE_NAME = ''
REPORT_MODEL_ID = ''
REPORT_ID = ''

COLLECT_CAPACITY_IDS = []
COLLECT_WORKSPACE_IDS = []
COLLECT_ITEM_TYPES = []
INCLUDE_PERSONAL_WORKSPACES = True
INCLUDE_DEEP_METADATA = True
INCLUDE_PERMISSIONS = True
INCLUDE_USAGE = True
EXPAND_GROUPS = True
ACTIVITY_BACKFILL_DAYS = 28

WRITE_MODE = "append_snapshot"
CONFIRM_OVERWRITE = False
TABLE_PREFIX = "inv_"
JOB_EXIT = False
VALIDATE_ALL_VIEWS = False


# Fabric Inventory Explorer: recipient guide

## Can I start with only the notebook?

**Yes. Upload `Fabric_Inventory_Template.ipynb`. No local Python files, Azure CLI, or deployment scripts are required.**

You still need a Fabric workspace and an active Fabric capacity. The notebook does not create, buy, resize, or resume capacity.

The notebook can create a dedicated lakehouse and, later, a semantic model and Power BI report.
These are separate, explicit actions. Uploading or opening the notebook does not create resources.

## Minimum setup

1. Upload the notebook to a suitable Fabric workspace. Select a supported **PySpark** runtime.
2. Read the first Markdown cells. The shared notebook starts with `MODE = "help"`.
3. Attach an existing **dedicated, schema-enabled inventory lakehouse** as the default.
4. If none exists, set `MODE = "setup_lakehouse"` and choose `SETUP_LAKEHOUSE_NAME`. Run the notebook.
5. Attach the returned lakehouse in the notebook Explorer. Restart the session so the default binding updates.
6. Set `MODE = "collect_and_view"` and `CONFIRM_WRITE_TARGET = True`. Run the notebook.
7. Return `MODE` to `"view_saved"` for ordinary browsing.

The notebook displays the resolved workspace and lakehouse before collection.
Do not confirm a business-data lakehouse as the inventory destination.

**Normally, leave all tenant, workspace, and lakehouse ID overrides blank.**
The notebook reads the attached-lakehouse context and the signed-in tenant.
Explicit overrides must match that context. They cannot redirect writes to another target.

## What should I configure?

| Setting | Configure it when | Meaning |
|---|---|---|
| `MODE` | Every time you choose a different action | `help`, `setup_lakehouse`, `view_saved`, `collect_and_view`, or `create_report` |
| `SETUP_LAKEHOUSE_NAME` | You need a new inventory lakehouse | A dedicated name, such as `FabricInventory`; existing items are not replaced |
| `CONFIRM_WRITE_TARGET` | Before the first collection | Confirms the displayed destination; it does **not** authorize overwrite |
| `REPORT_MODEL_NAME`, `REPORT_NAME` | You want the optional Power BI output | Choose unique names if unrelated items already use the defaults |
| `COLLECT_WORKSPACE_IDS`, `COLLECT_CAPACITY_IDS`, `COLLECT_ITEM_TYPES` | You need a narrower scan | Collection filters, not viewing filters |
| `INCLUDE_DEEP_METADATA`, `INCLUDE_PERMISSIONS`, `INCLUDE_USAGE` | You want to change collection depth | Defaults collect each supported category, subject to access and service limits |
| `EXPAND_GROUPS`, `GRAPH_CREDENTIAL` | Approved Graph access is already available | Optional group expansion; no new credentials or consent are created |
| `TABLE_PREFIX` | You need a separate inventory table set | Set it before the first collection; the report uses the same prefix |
| `WRITE_MODE` | Normally leave unchanged | `append_snapshot` preserves history |
| `CONFIRM_OVERWRITE` | Only for an intentional destructive reset | Must be Boolean `True` together with `WRITE_MODE = "replace_inventory"` |
| Tenant/workspace/lakehouse overrides | Advanced use only | Normally blank; incorrect overrides stop the action with a destination-mismatch message |
| `REPORT_MODEL_ID`, `REPORT_ID` | Advanced reuse of an existing inventory deployment | Verify the IDs first; `create_report` never overwrites existing definitions |

`VIEW` controls affect saved-data exploration only. Changing them does not collect data or write tables.

## Create the optional report from the notebook

1. Complete a collection so a published snapshot exists.
2. Set `MODE = "create_report"`.
3. Review `REPORT_MODEL_NAME` and `REPORT_NAME`, then run the notebook.
4. Open the returned report link. Return `MODE` to `"view_saved"` afterward.

The action creates a Direct Lake semantic model, initializes it, and creates the report in the inventory lakehouse workspace.
All report/model definitions are embedded in the notebook.

The report is optional. Notebook views remain useful if report creation is denied.
Missing report rights do not erase inventory data or turn a failed report creation into a reported success.

Repeated report setup reuses items identified as this inventory's managed deployment.
An unrelated name collision stops setup. It does not replace or rebind that item.
If only the model was created before an error, its ID is reported for recovery.

## Which rights are required?

| Operation | Typical requirements | Important distinction |
|---|---|---|
| Read the instructions | Access to the notebook | No source inventory is queried |
| Run the notebook | Notebook execution rights and an available Fabric capacity | Workspace Contributor or higher is the straightforward authoring route |
| Browse saved inventory | Read access to the inventory lakehouse and notebook execution rights | Fabric tenant-admin rights are not needed |
| Create the lakehouse or write inventory | Suitable authoring/write rights in the destination workspace/lakehouse | Azure subscription Owner does not grant Fabric workspace access |
| Tenant-wide admin inventory | Fabric administrator role and a suitable delegated admin token | Workspace Admin is not Fabric administrator |
| Accessible-workspace fallback | Access to those workspaces/items | This is not a complete tenant inventory |
| Deep model/table metadata | Workload-specific read/metadata access; model Read/Build where required | Admin catalog access does not automatically grant data-plane access |
| Activity and tenant-wide permission APIs | Their admin role and token scopes | If unavailable, usage/permissions are labeled unavailable, not zero |
| Create a Power BI model/report | Destination authoring rights, an appropriate Pro/PPU/trial license, and source access | A capacity does not replace the report author's license |
| Expand groups | An existing supported Graph credential and approved membership-read permissions | A Fabric role does not grant Microsoft Graph permissions |

For ordinary Power BI viewing on an F capacity below F64, viewers also need an appropriate per-user license.
Default Direct Lake SSO requires the viewer to have suitable access to the inventory data.

## What happens when rights are missing?

- **Admin API denied:** permitted standard APIs are used where an equivalent exists. Scope labels show the reduced coverage.
- **Source item denied:** the catalog item remains visible. Its missing detail is not shown as an empty source.
- **Token scope missing:** the message identifies an authentication-scope issue. Adding a workspace role alone may not solve it.
- **Sign-in expired or wrong tenant:** the action stops. Sign in to the correct Fabric tenant and renew the session.
- **Capacity paused:** the notebook explains the unavailable detail. It does not resume the source.
- **API throttled:** remaining entities are deferred with a retry explanation. Previous snapshots remain intact.
- **Resource not found:** check both existence and visibility. A 404 is not proof of deletion.
- **Report creation denied:** the error names the failed action and required authoring/license checks. Any created model is identified.

Use **Diagnostics** for the collector, entity, HTTP code, service code, reason, and recommended next step.
Never enable overwrite to resolve an access problem.

## Share the template, not the results

Distribute only the clean template notebook and this guide from the external-sharing package.
The generated template has no tenant IDs, resource IDs, local paths, runtime outputs, or inventory records.
Review the code before running it. Package checksums detect changes; they do not authenticate the publisher.

Do **not** distribute the internal deployment ZIP, configured notebook, `deployment.json`, live validation files, or inventory lakehouse.
An executed notebook can retain names, user identifiers, permissions, and usage in its outputs.
Clearing outputs alone does not remove configured IDs, attached-lakehouse metadata, or embedded report filters.

If you intend to share actual inventory results, treat that as a separate access and privacy review.
Tenant metadata, identities, group membership, and usage can be sensitive.
Keep raw inventory restricted; share a minimized, approved extract when appropriate.
Power BI RLS does not protect raw lakehouse tables from people who can read them directly.
Guest access, licensing, tenant settings, and source-data permissions need separate review.

This package does not publish to the web, add guests, grant roles, or share a workspace.

## Limits and operating model

- Collection is manual. No schedule or pause/resume automation is created.
- Activity retention is rolling and limited. Older complete UTC days may fall outside the service window.
- MAU is based on observed, identifiable actors. Incomplete windows remain labeled partial.
- Permission rows are observed assignments, not a full SQL/RLS/OLS/OneLake effective-access audit.
- Snapshots preserve history. Comparisons require compatible scope and coverage.
- Do not run multiple writers against the same inventory destination.
- Package versions, service capabilities, and Preview APIs can change. Verify in a limited workspace before wider use.

In [ ]:
VIEW = {
    "view": "Overview",
    "run_id": "",
    "baseline_run_id": "",
    "capacity_id": "",
    "workspace_id": "",
    "item_id": "",
    "item_type": "",
    "principal_id": "",
    "permission": "",
    "domain_id": "",
    "tag": "",
    "state": "",
    "search": "",
    "start_date": "",
    "end_date": "",
    "collection": "items",
    "page": 1,
    "page_size": 100,
}

# Optional existing supported Azure TokenCredential for Microsoft Graph.
# This notebook does not request new consent or embed a local CLI token.
GRAPH_CREDENTIAL = None


## Implementation helpers

These collapsed cells contain the collection contracts, destination safeguards, and read-only views.
They are embedded here so the notebook has no dependency on local Python files.
Loading the definitions performs no authentication, source queries, or writes.
The collection path uses the Semantic Link clients included in Fabric.
It does not upgrade packages automatically.

In [ ]:
"""Fabric inventory contracts, collectors, and pure transformations.

Embedded into Fabric_Inventory.ipynb by build_notebook.py.
No authentication, network requests, or writes occur when this module is loaded.
"""

import hashlib
import json
import re
import time
import uuid
from collections import Counter
from datetime import date, datetime, timedelta, timezone
from urllib.parse import parse_qsl, urlencode, urljoin, urlparse, urlunparse


FABRIC = "https://api.fabric.microsoft.com"
POWERBI = "https://api.powerbi.com"
GRAPH = "https://graph.microsoft.com"
SCHEMA_VERSION = "1"
COMMON = {
    "row_id": "string", "tenant_id": "string", "run_id": "string",
    "collected_at_utc": "string", "source_api": "string",
    "source_scope": "string", "schema_version": "string",
    "capacity_key": "string", "workspace_key": "string", "item_key": "string",
}
TABLES = {
    "capacities": {
        "capacity_id": "string", "capacity_name": "string", "sku": "string",
        "region": "string", "state": "string",
    },
    "workspaces": {
        "workspace_id": "string", "workspace_name": "string", "description": "string",
        "workspace_type": "string", "state": "string", "capacity_id": "string",
        "domain_id": "string", "license_mode": "string", "url": "string",
    },
    "items": {
        "workspace_id": "string", "item_id": "string", "item_name": "string",
        "item_type": "string", "description": "string", "state": "string",
        "capacity_id": "string", "creator_id": "string", "creator_name": "string",
        "owner_name": "string", "modified_at_utc": "string", "folder_id": "string",
        "endorsement": "string", "sensitivity_label_id": "string", "url": "string",
    },
    "domains": {
        "domain_id": "string", "domain_name": "string", "description": "string",
        "parent_domain_id": "string",
    },
    "tags": {
        "entity_id": "string", "workspace_id": "string", "tag_id": "string",
        "tag_name": "string", "entity_type": "string",
    },
    "reports": {
        "workspace_id": "string", "item_id": "string", "report_name": "string",
        "report_type": "string", "model_id": "string", "model_workspace_id": "string",
        "url": "string",
    },
    "semantic_models": {
        "workspace_id": "string", "item_id": "string", "model_name": "string",
        "configured_by": "string", "is_refreshable": "boolean", "storage_mode": "string",
        "schema_error": "string",
    },
    "tables": {
        "workspace_id": "string", "item_id": "string", "table_id": "string",
        "table_name": "string", "schema_name": "string", "parent_type": "string",
        "table_type": "string", "storage_mode": "string", "location": "string",
    },
    "columns": {
        "workspace_id": "string", "item_id": "string", "table_id": "string",
        "table_name": "string", "column_name": "string", "data_type": "string",
        "is_hidden": "boolean", "is_nullable": "boolean",
    },
    "measures": {
        "workspace_id": "string", "item_id": "string", "table_name": "string",
        "measure_name": "string", "data_type": "string", "format_string": "string",
        "description": "string", "is_hidden": "boolean",
    },
    "relationships": {
        "workspace_id": "string", "item_id": "string", "relationship_name": "string",
        "from_table": "string", "from_column": "string", "to_table": "string",
        "to_column": "string", "is_active": "boolean", "cross_filter": "string",
    },
    "data_sources": {
        "workspace_id": "string", "item_id": "string", "datasource_id": "string",
        "datasource_type": "string", "server": "string", "database_name": "string",
        "gateway_id": "string",
    },
    "lineage": {
        "workspace_id": "string", "item_id": "string", "source_item_id": "string",
        "source_workspace_id": "string", "target_item_id": "string",
        "target_workspace_id": "string", "relationship_type": "string",
        "source_name": "string", "target_name": "string",
    },
    "endpoints": {
        "workspace_id": "string", "item_id": "string", "endpoint_id": "string",
        "endpoint_type": "string", "server": "string", "state": "string",
    },
    "shortcuts": {
        "workspace_id": "string", "item_id": "string", "shortcut_name": "string",
        "path": "string", "target_type": "string", "target_workspace_id": "string",
        "target_item_id": "string", "target_path": "string",
    },
    "principals": {
        "principal_id": "string", "principal_name": "string", "principal_type": "string",
        "user_principal_name": "string",
    },
    "access_assignments": {
        "capacity_id": "string", "workspace_id": "string", "item_id": "string",
        "entity_id": "string", "entity_name": "string", "entity_type": "string",
        "principal_id": "string", "principal_name": "string", "principal_type": "string",
        "permission": "string", "permission_kind": "string", "assignment_scope": "string",
        "inheritance": "string",
    },
    "group_memberships": {
        "group_id": "string", "member_id": "string", "member_name": "string",
        "member_type": "string", "user_principal_name": "string", "membership_kind": "string",
    },
    "refresh_status": {
        "workspace_id": "string", "item_id": "string", "request_id": "string",
        "refresh_type": "string", "state": "string", "started_at_utc": "string",
        "ended_at_utc": "string",
    },
    "report_view_events": {
        "event_id": "string", "event_at_utc": "string", "event_date": "string",
        "workspace_id": "string", "item_id": "string", "actor_id": "string",
        "actor_kind": "string", "is_human": "boolean", "success": "boolean",
        "distribution_method": "string", "consumption_method": "string",
        "report_name": "string", "workspace_name": "string",
    },
    "calendar": {
        "date_key": "string", "date": "date", "year": "long",
        "month_number": "long", "month_name": "string",
    },
    "activity_day_status": {
        "event_date": "string", "status": "string", "event_count": "long",
        "window_start_utc": "string", "window_end_utc": "string",
    },
    "collection_status": {
        "collector": "string", "entity_id": "string", "workspace_id": "string",
        "item_id": "string", "status": "string", "row_count": "long",
        "http_status": "long", "error_code": "string", "message": "string",
    },
    "run_events": {
        "event": "string", "status": "string", "message": "string",
        "scope_fingerprint": "string",
    },
    "published_runs": {
        "published_at_utc": "string", "scope_fingerprint": "string",
        "status": "string", "workspace_count": "long", "item_count": "long",
        "coverage_gap_count": "long", "usage_through_date": "string",
        "package_version": "string",
        "catalog_scope": "string",
    },
    "changes": {
        "baseline_run_id": "string", "entity_type": "string", "entity_id": "string",
        "workspace_id": "string", "item_id": "string", "change_type": "string",
        "field_name": "string", "before_value": "string", "after_value": "string",
    },
    "findings": {
        "workspace_id": "string", "item_id": "string", "finding_type": "string",
        "summary": "string", "evidence": "string",
    },
}


def utc_now():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def identifier(value):
    return str(value).strip().lower() if value is not None else ""


def stable_id(*parts):
    text = json.dumps(parts, sort_keys=True, ensure_ascii=True, default=str)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def safe_text(value, limit=2000):
    if value is None:
        return None
    text = str(value)
    text = re.sub(r"(?i)(bearer\s+)[^\s,;]+", r"\1[redacted]", text)
    text = re.sub(r"(?i)(password|client[_-]?secret|access[_-]?token|sharedaccesssignature|sig)\s*[:=]\s*([^&\s;]+)",
                  r"\1=[redacted]", text)
    text = re.sub(r"\beyJ[A-Za-z0-9_-]+\.[A-Za-z0-9_-]+\.[A-Za-z0-9_-]+", "[redacted-jwt]", text)
    return text[:limit]


def server_metadata(value):
    """Keep a server host, never a connection string or credential-bearing URL."""
    if value is None:
        return None
    text = str(value).strip()
    if "=" in text:
        values = dict(part.split("=", 1) for part in text.split(";") if "=" in part)
        text = next((v.strip() for k, v in values.items() if k.strip().lower() in {"server", "data source", "address"}), "")
    if "://" in text:
        return urlparse(text).hostname
    text = text.removeprefix("tcp:")
    return text if re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]*(?:,[0-9]+)?", text) else None


def required(record, key):
    value = record.get(key)
    if value is None or value == "":
        raise ValueError("Required API field is missing: " + key)
    return value


def validate_config(config):
    if config["mode"] not in {"view_saved", "collect_and_view"}:
        raise ValueError("MODE must be view_saved or collect_and_view.")
    if config["write_mode"] not in {"append_snapshot", "replace_inventory"}:
        raise ValueError("WRITE_MODE must be append_snapshot or replace_inventory.")
    if config["mode"] == "collect_and_view":
        if config["write_mode"] == "replace_inventory" and config["confirm_overwrite"] is not True:
            raise PermissionError("Replacement requires Boolean CONFIRM_OVERWRITE = True before any write.")
    if not re.fullmatch(r"[A-Za-z][A-Za-z0-9_]*", config["table_prefix"]):
        raise ValueError("TABLE_PREFIX must be a safe SQL identifier.")
    if not 1 <= config["activity_days"] <= 28:
        raise ValueError("ACTIVITY_BACKFILL_DAYS must be between 1 and 28.")
    uuid.UUID(config["tenant_id"])
    for key in ("capacity_ids", "workspace_ids"):
        for value in config[key]:
            uuid.UUID(value)
    return config


def scope_fingerprint(config, scopes):
    return stable_id({
        "tenant": identifier(config["tenant_id"]),
        "capacities": sorted(map(identifier, config["capacity_ids"])),
        "workspaces": sorted(map(identifier, config["workspace_ids"])),
        "types": sorted(config["item_types"]),
        "personal": config["include_personal"],
        "scopes": scopes,
    })


class ApiFailure(RuntimeError):
    def __init__(self, status, code, message, url=""):
        self.status = status
        self.code = code
        self.url = url
        super().__init__(safe_text(message))


def collection_guidance(collector, status, scope, reason=""):
    if status in {"complete", "empty", "not_requested"}:
        return ""
    text = reason.lower()
    if status == "paused":
        return ("Ask the capacity administrator to confirm source availability. Resuming can incur charges and is not automatic. "
                "Catalog rows remain visible; missing detail is not an empty source.")
    if status == "throttled":
        return ("Wait for the service retry window or narrow the collection filters. "
                "Deferred entities are marked unavailable, not zero. Existing snapshots are preserved.")
    if status == "outside_retention":
        return ("This UTC day is outside the activity API's rolling retention window. "
                "Collect regularly and retain the archive; do not interpret incomplete MAU as a full month.")
    if status == "unresolved" or collector == "group_members":
        return ("Groups remain listed without verified members. Use an existing, approved Graph credential with group-membership read access. "
                "No new consent or directory role is granted by this notebook.")
    if status == "denied":
        if "scope" in text:
            return ("The token is missing an API scope. A higher workspace role alone may not fix this. "
                    "Ask your identity administrator about approved authentication/consent, or use the available notebook-only views.")
        if scope == "admin":
            return ("Tenant-wide collection requires the Fabric administrator role and an appropriate delegated admin token. "
                    "Workspace Admin and Azure subscription Owner are not equivalent. Accessible-workspace fallback is labeled where available.")
        if collector.startswith("model_"):
            return ("Ask the semantic-model owner to confirm Read/Build access and an available XMLA-capable source. "
                    "Admin catalog access does not grant model data access.")
        return ("Ask the workspace/item owner to confirm the read or metadata permissions needed by this API. "
                "The collection is unavailable, not empty; no access is granted automatically.")
    if status in {"unsupported", "unverified_capacity"}:
        return ("Check the item type, source capacity, and visibility in Fabric. A 404 can also conceal an inaccessible item. "
                "Do not infer deletion or zero permissions from this result.")
    return ("Review the service code and collector in Diagnostics. Correct the source or configuration, then rerun. "
            "Do not enable overwrite merely to resolve a collection error.")


class RestApi:
    """Client around SemPy or requests-compatible authenticated transports."""

    def __init__(self, base_url, transport, sleep=time.sleep, hourly_limit=190):
        self.base_url = base_url.rstrip("/")
        self.transport = transport
        self.sleep = sleep
        self.hourly_limit = hourly_limit
        self.calls = {}
        self.cooldowns = {}

    def url(self, path):
        url = urljoin(self.base_url + "/", path)
        parsed = urlparse(url)
        allowed_path = "/v1/" if self.base_url == FABRIC else "/v1.0/myorg/"
        if (self.base_url in (FABRIC, POWERBI) and parsed.scheme == "https"
                and re.fullmatch(r"wabi-[a-z0-9-]+\.analysis\.windows\.net", parsed.hostname or "")
                and parsed.port in (None, 443) and parsed.path.startswith(allowed_path)):
            # Route regional service-issued locations through the public gateway.
            # Credentials are never forwarded to the regional hostname.
            url = urlunparse(parsed._replace(netloc=urlparse(self.base_url).netloc))
            parsed = urlparse(url)
        if parsed.username or parsed.password:
            raise ValueError("User information is not allowed in API URLs.")
        if parsed.hostname == urlparse(self.base_url).hostname and parsed.port in (None, 443):
            url = urlunparse(parsed._replace(netloc=urlparse(self.base_url).netloc))
            parsed = urlparse(url)
        if parsed.scheme != "https" or parsed.netloc != urlparse(self.base_url).netloc:
            raise ValueError("Refusing a cross-host continuation or authentication redirect.")
        return url

    def request(self, method, path, family=None, **kwargs):
        import requests
        url = self.url(path)
        family = family or urlparse(url).path
        for attempt in range(5):
            now = time.monotonic()
            if now < self.cooldowns.get(family, 0):
                raise ApiFailure(429, "EndpointCooldown", "Endpoint retry window is still active; remaining entities are deferred.", url)
            calls = [stamp for stamp in self.calls.get(family, []) if now - stamp < 3600]
            if len(calls) >= self.hourly_limit:
                raise ApiFailure(429, "LocalRequestBudget", "Endpoint request budget reached; rerun later.", url)
            calls.append(now)
            self.calls[family] = calls
            try:
                response = self.transport(method, url, **kwargs)
            except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as error:
                if method == "GET" and attempt < 4:
                    self.sleep(2 ** attempt)
                    continue
                raise ApiFailure(503, "TransportError", "Request failed after bounded transport retries.", url) from error
            if response.status_code in (429, 500, 502, 503, 504) and method == "GET" and attempt < 4:
                header = response.headers.get("Retry-After", "")
                wait = max(2 ** attempt, int(header) if header.isdigit() else 0)
                if wait > 120:
                    if response.status_code == 429:
                        self.cooldowns[family] = now + wait
                    raise ApiFailure(response.status_code, "RetryLater", "Service asks for a longer retry delay.", url)
                self.sleep(wait)
                continue
            if not 200 <= response.status_code < 300:
                if response.status_code == 429:
                    header = response.headers.get("Retry-After", "")
                    self.cooldowns[family] = now + (int(header) if header.isdigit() else 3600)
                try:
                    body = response.json()
                except ValueError:
                    body = {}
                error = body.get("error", body)
                if not isinstance(error, dict):
                    error = {}
                message = error.get("message", error.get("errorMessage"))
                if not message:
                    pbi = error.get("pbi.error", {})
                    details = pbi.get("details", []) if isinstance(pbi, dict) else []
                    messages = [d.get("detail", {}).get("value") for d in details if d.get("code") == "DetailsMessage"]
                    message = "; ".join(str(value) for value in messages if value) or "API request failed."
                raise ApiFailure(
                    response.status_code, error.get("code", error.get("errorCode", "HttpError")),
                    message, url,
                )
            return response
        raise RuntimeError("Unreachable retry state.")

    def get(self, path, family=None):
        return self.request("GET", path, family=family).json()

    def pages(self, path, keys=("value",), family=None):
        origin = self.url(path)
        next_url = origin
        seen = set()
        while next_url:
            if next_url in seen:
                raise ValueError("Repeated continuation URL; pagination is incomplete.")
            seen.add(next_url)
            body = self.get(next_url, family)
            key = next((key for key in keys if key in body), None)
            if key is None or not isinstance(body[key], list):
                raise ValueError(
                    "Unexpected collection response at " + urlparse(next_url).path
                    + "; expected " + "/".join(keys) + "; received keys " + ", ".join(sorted(body))
                )
            for row in body[key]:
                if not isinstance(row, dict):
                    raise ValueError("Collection row is not an object.")
                yield row
            next_url = body.get("continuationUri") or body.get("@odata.nextLink") or body.get("nextLink")
            token = body.get("continuationToken")
            if not next_url and token:
                parsed = urlparse(origin)
                query = dict(parse_qsl(parsed.query))
                if parsed.path.endswith("/admin/activityevents"):
                    query.pop("startDateTime", None)
                    query.pop("endDateTime", None)
                query["continuationToken"] = token
                next_url = urlunparse(parsed._replace(query=urlencode(query)))
            if next_url:
                next_url = self.url(next_url)


class Inventory:
    def __init__(self, tenant_id, run_id=None, stamp=None):
        self.tenant_id = identifier(tenant_id)
        self.run_id = run_id or str(uuid.uuid4())
        self.stamp = stamp or utc_now()
        self.rows = {name: {} for name in TABLES}
        self.scopes = {}

    def add(self, table, values, source, scope, key=None):
        fields = TABLES[table]
        unknown = set(values) - set(fields)
        if unknown:
            raise ValueError(f"Unknown {table} fields: {sorted(unknown)}")
        clean = {}
        for name, kind in fields.items():
            value = values.get(name)
            if value is not None:
                if kind == "string":
                    value = str(value)
                elif kind == "boolean" and not isinstance(value, bool):
                    raise ValueError(f"{table}.{name} must be Boolean.")
                elif kind == "long":
                    value = int(value)
                elif kind == "date" and not isinstance(value, date):
                    value = date.fromisoformat(value)
            clean[name] = value
        identity = stable_id(self.tenant_id, self.run_id, table, key if key is not None else clean)
        result = dict(
            row_id=identity, tenant_id=self.tenant_id, run_id=self.run_id,
            collected_at_utc=self.stamp, source_api=source, source_scope=scope,
            schema_version=SCHEMA_VERSION, **clean,
        )
        for entity in ("capacity", "workspace", "item"):
            entity_id = clean.get(entity + "_id")
            parts = [self.tenant_id, self.run_id, entity_id]
            if entity == "item":
                parts.insert(2, clean.get("workspace_id"))
            result[entity + "_key"] = stable_id(*parts) if entity_id else None
        self.rows[table][identity] = result
        return result

    def data(self, table):
        return list(self.rows[table].values())

    def status(self, collector, entity, status, count=0, api="", scope="admin",
               error=None, workspace_id="", item_id="", message=""):
        reason = safe_text(str(error) if error else message, 700) or ""
        guidance_reason = (str(getattr(error, "code", "")) + " " + reason).strip()
        guidance = collection_guidance(collector, status, scope, guidance_reason)
        message = reason + (("\nNext step: " + guidance) if guidance else "")
        return self.add("collection_status", {
            "collector": collector, "entity_id": entity, "workspace_id": workspace_id,
            "item_id": item_id, "status": status, "row_count": count,
            "http_status": getattr(error, "status", None),
            "error_code": getattr(error, "code", None),
            "message": safe_text(message),
        }, api, scope, key=(collector, entity, api))


def failure_kind(error):
    message = str(error).lower()
    if "capacitynotactive" in message or ("capacity" in message and any(word in message for word in ("paused", "suspended", "not active", "not available"))):
        return "paused"
    if error.status == 403:
        return "denied"
    if error.status == 429:
        return "throttled"
    if error.status in (404, 405, 501):
        return "unsupported"
    if "not supported" in message or "only be called on a model-based dataset" in message:
        return "unsupported"
    if "capacity" in error.code.lower() and "inactive" in message:
        return "paused"
    return "failed"


class Collector:
    def __init__(self, fabric, powerbi, config, inventory=None, graph=None):
        self.fabric, self.powerbi, self.graph = fabric, powerbi, graph
        self.config = validate_config(config)
        self.inv = inventory or Inventory(config["tenant_id"])
        self.workspace_map = {}
        self.capacity_map = {}
        self.item_map = {}
        self.datasource_index = {}
        self.auth_probes = {}

    def recoverable_failure(self, error, api):
        if error.status != 401:
            return failure_kind(error)
        if error.code in {"InvalidAuthenticationToken", "ExpiredAuthenticationToken", "TokenExpired", "InvalidToken"}:
            raise error
        probes = {
            FABRIC: "/v1/workspaces",
            POWERBI: "/v1.0/myorg/groups?$top=1",
        }
        if api.base_url not in probes:
            raise error
        now = time.monotonic()
        if now - self.auth_probes.get(api.base_url, -3600) > 60:
            api.get(probes[api.base_url], family="authentication_probe")
            self.auth_probes[api.base_url] = now
        return "denied"

    def collect_rows(self, name, entity, api, path, keys=("value",), scope="admin",
                     workspace_id="", item_id=""):
        try:
            values = list(api.pages(path, keys, family=name))
        except ApiFailure as error:
            kind = self.recoverable_failure(error, api)
            self.inv.status(name, entity, kind, api=path, scope=scope,
                            error=error, workspace_id=workspace_id, item_id=item_id)
            return None
        self.inv.status(name, entity, "complete" if values else "empty", len(values),
                        path, scope, workspace_id=workspace_id, item_id=item_id)
        return values

    def admin_or_user(self, name, api, path, keys, fallback_api, fallback_path, fallback_keys, allow_unavailable=False):
        try:
            values = list(api.pages(path, keys, family=name))
            scope = "admin"
        except ApiFailure as error:
            if error.status == 401:
                kind = self.recoverable_failure(error, api)
            elif error.status in (403, 404, 405, 501):
                kind = failure_kind(error)
            else:
                raise
            self.inv.status(name + "_admin", self.inv.tenant_id, kind,
                            api=path, error=error)
            try:
                values = list(fallback_api.pages(fallback_path, fallback_keys, family=name))
            except ApiFailure as fallback_error:
                if not allow_unavailable or fallback_error.status not in (403, 404, 405, 501):
                    raise
                self.inv.scopes[name] = "unavailable"
                self.inv.status(name, self.inv.tenant_id, "unavailable", api=fallback_path, scope="user",
                                error=fallback_error)
                return [], "unavailable"
            scope = "user"
        self.inv.scopes[name] = scope
        self.inv.status(name, self.inv.tenant_id, "complete" if values else "empty",
                        len(values), path if scope == "admin" else fallback_path, scope)
        return values, scope

    def add_tags(self, raw, entity_id, wsid, entity_type, source, scope):
        for tag in raw.get("tags") or []:
            self.inv.add("tags", {
                "entity_id": entity_id, "workspace_id": wsid, "entity_type": entity_type,
                "tag_id": identifier(tag.get("id")),
                "tag_name": tag.get("displayName", tag.get("name", "")),
            }, source, scope)

    def add_principal(self, principal, source, scope):
        pid = identifier(principal.get("id") or principal.get("graphId") or principal.get("identifier")
                         or principal.get("emailAddress"))
        if not pid:
            raise ValueError("Permission entry has no principal identifier.")
        ptype = principal.get("type", principal.get("principalType", "Unknown"))
        result = {
            "principal_id": pid, "principal_name": principal.get("displayName", principal.get("emailAddress", pid)),
            "principal_type": ptype,
            "user_principal_name": (principal.get("userDetails") or {}).get(
                "userPrincipalName", principal.get("emailAddress", principal.get("userPrincipalName"))),
        }
        self.inv.add("principals", result, source, scope, key=pid)
        return result

    def catalog(self):
        capacities, scope = self.admin_or_user(
            "capacities", self.powerbi, "/v1.0/myorg/admin/capacities", ("value",),
            self.fabric, "/v1/capacities", ("value",), allow_unavailable=True,
        )
        for row in capacities:
            cid = identifier(required(row, "id"))
            self.capacity_map[cid] = row
            self.inv.add("capacities", {
                "capacity_id": cid, "capacity_name": row.get("displayName", row.get("name")),
                "sku": row.get("sku"), "region": row.get("region"), "state": row.get("state"),
            }, "capacities", scope, key=cid)
            for user in row.get("admins") or []:
                principal = self.add_principal(
                    {"id": user, "displayName": user, "type": "User", "userPrincipalName": user},
                    "capacity_admins", scope)
                self.inv.add("access_assignments", {
                    **{k: principal[k] for k in ("principal_id", "principal_name", "principal_type")},
                    "capacity_id": cid, "entity_id": cid, "entity_name": row.get("displayName"),
                    "entity_type": "Capacity", "permission": "Admin", "permission_kind": "role",
                    "assignment_scope": "capacity", "inheritance": "reported",
                }, "capacity_admins", scope)
        workspaces, scope = self.admin_or_user(
            "workspaces", self.fabric, "/v1/admin/workspaces", ("workspaces",),
            self.fabric, "/v1/workspaces", ("value",),
        )
        allowed_c = set(map(identifier, self.config["capacity_ids"]))
        allowed_w = set(map(identifier, self.config["workspace_ids"]))
        for row in workspaces:
            wsid = identifier(required(row, "id"))
            cid = identifier(row.get("capacityId"))
            if allowed_c and cid not in allowed_c or allowed_w and wsid not in allowed_w:
                continue
            if not self.config["include_personal"] and row.get("type", "").lower() == "personal":
                continue
            self.workspace_map[wsid] = row
            self.inv.add("workspaces", {
                "workspace_id": wsid, "workspace_name": row.get("name", row.get("displayName")),
                "description": row.get("description"), "workspace_type": row.get("type"),
                "state": row.get("state", "Unknown"), "capacity_id": cid,
                "domain_id": identifier(row.get("domainId")),
                "license_mode": row.get("licenseMode", "Unknown"),
                "url": f"https://app.fabric.microsoft.com/groups/{wsid}/list",
            }, "workspaces", scope, key=wsid)
            self.add_tags(row, wsid, wsid, "Workspace", "workspaces", scope)
        item_scope = "admin"
        try:
            items = list(self.fabric.pages("/v1/admin/items", ("itemEntities",), "items"))
            self.inv.status("items", self.inv.tenant_id, "complete" if items else "empty",
                            len(items), "/v1/admin/items")
        except ApiFailure as error:
            if error.status == 401:
                kind = self.recoverable_failure(error, self.fabric)
            elif error.status in (403, 404, 405, 501):
                kind = failure_kind(error)
            else:
                raise
            self.inv.status("items_admin", self.inv.tenant_id, kind,
                            api="/v1/admin/items", error=error)
            item_scope = "user"
            items = []
            for wsid, ws in self.workspace_map.items():
                if ws.get("state", "Active").lower() != "active":
                    continue
                values = self.collect_rows("items_user", wsid, self.fabric,
                                           f"/v1/workspaces/{wsid}/items", scope="user", workspace_id=wsid)
                for item in values or []:
                    items.append(dict(item, workspaceId=wsid))
            self.inv.status("items", self.inv.tenant_id, "partial", len(items),
                            "workspace_items", "user", message="Only accessible workspace items are included.")
        self.inv.scopes["items"] = item_scope
        for row in items:
            wsid = identifier(required(row, "workspaceId"))
            if wsid not in self.workspace_map:
                continue
            self.add_item(row, wsid, "admin_items" if item_scope == "admin" else "workspace_items", item_scope)
        domains = self.collect_rows("domains", self.inv.tenant_id, self.fabric,
                                    "/v1/admin/domains", ("domains", "value"))
        for row in domains or []:
            self.inv.add("domains", {
                "domain_id": identifier(required(row, "id")),
                "domain_name": row.get("displayName", row.get("name")),
                "description": row.get("description"),
                "parent_domain_id": identifier(row.get("parentDomainId")),
            }, "admin_domains", "admin", key=identifier(row["id"]))
        return self.inv

    def add_item(self, row, wsid, source, scope, item_type=None):
        item_id = identifier(required(row, "id"))
        kind = item_type or required(row, "type")
        if self.config["item_types"] and kind not in self.config["item_types"]:
            return None
        ws = self.workspace_map[wsid]
        old = self.item_map.get(item_id, {})
        combined = {**old, **{k: v for k, v in row.items() if v is not None}}
        combined["type"] = kind
        combined["workspaceId"] = wsid
        self.item_map[item_id] = combined
        creator = combined.get("creatorPrincipal") or {}
        result = self.inv.add("items", {
            "workspace_id": wsid, "item_id": item_id,
            "item_name": combined.get("name", combined.get("displayName")), "item_type": kind,
            "description": combined.get("description"), "state": combined.get("state", "Active"),
            "capacity_id": identifier(combined.get("capacityId", ws.get("capacityId"))),
            "creator_id": identifier(creator.get("id")), "creator_name": creator.get("displayName"),
            "owner_name": combined.get("configuredBy"),
            "modified_at_utc": combined.get("lastUpdatedDate", combined.get("modifiedDateTime")),
            "folder_id": identifier(combined.get("folderId")),
            "endorsement": (combined.get("endorsementDetails") or {}).get("endorsement"),
            "sensitivity_label_id": (combined.get("sensitivityLabel") or {}).get("labelId"),
            "url": combined.get("webUrl") or f"https://app.fabric.microsoft.com/groups/{wsid}/list?itemId={item_id}",
        }, source, scope, key=(wsid, item_id))
        self.add_tags(combined, item_id, wsid, kind, source, scope)
        return result

    def scanner(self):
        if not self.config["include_deep"]:
            self.inv.status("scanner", self.inv.tenant_id, "not_requested")
            return
        workspaces = [w for w, row in self.workspace_map.items() if row.get("state", "Active").lower() == "active"]
        path = "/v1.0/myorg/admin/workspaces/getInfo?" + urlencode({
            "lineage": "true", "datasourceDetails": "true",
            "datasetSchema": "true", "datasetExpressions": "false",
            "getArtifactUsers": "true" if self.config["include_permissions"] else "false",
        })
        for start in range(0, len(workspaces), 100):
            batch = workspaces[start:start + 100]
            try:
                response = self.powerbi.request("POST", path, family="scanner_start", json={"workspaces": batch})
                scan_id = required(response.json(), "id")
                deadline = time.monotonic() + 1200
                while True:
                    state = self.powerbi.get(f"/v1.0/myorg/admin/workspaces/scanStatus/{scan_id}", "scanner_status")
                    if state.get("status") == "Succeeded":
                        break
                    if state.get("status") in ("Failed", "Cancelled"):
                        raise ApiFailure(422, "ScanFailed", "Metadata scan did not complete.")
                    if time.monotonic() >= deadline:
                        raise ApiFailure(408, "ScanTimeout", "Metadata scan exceeded its polling window.")
                    self.powerbi.sleep(30)
                result = self.powerbi.get(f"/v1.0/myorg/admin/workspaces/scanResult/{scan_id}", "scanner_result")
                if "workspaces" not in result:
                    raise ValueError("Scan result has no workspaces collection.")
                self.datasource_index.update({
                    identifier(source["datasourceId"]): source
                    for source in result.get("datasourceInstances") or []
                    if source.get("datasourceId")
                })
                returned = set()
                for workspace in result["workspaces"]:
                    wsid = identifier(required(workspace, "id"))
                    returned.add(wsid)
                    self.parse_scan(workspace)
                    self.inv.status("scanner", wsid, "complete", 1, path, workspace_id=wsid)
                for wsid in set(batch) - returned:
                    self.inv.status("scanner", wsid, "failed", api=path, workspace_id=wsid,
                                    message="Requested workspace missing from scan result.")
            except ApiFailure as error:
                kind = self.recoverable_failure(error, self.powerbi)
                for wsid in batch:
                    self.inv.status("scanner", wsid, kind, api=path,
                                    error=error, workspace_id=wsid)

    def parse_scan(self, workspace):
        wsid = identifier(required(workspace, "id"))
        if wsid not in self.workspace_map:
            raise ValueError("Scanner returned a workspace outside the requested scope.")
        for existing in self.inv.data("workspaces"):
            if existing["workspace_id"] == wsid:
                existing["description"] = workspace.get("description") or existing["description"]
        self.add_tags(workspace, wsid, wsid, "Workspace", "scanner", "admin")
        for plural, kind in (("reports", "Report"), ("datasets", "SemanticModel"),
                             ("dashboards", "Dashboard"), ("dataflows", "Dataflow"),
                             ("datamarts", "Datamart")):
            for row in workspace.get(plural) or []:
                item = self.add_item(row, wsid, "scanner", "admin", kind)
                if item is None:
                    continue
                iid = item["item_id"]
                if kind == "Report":
                    self.add_report(row, wsid, "scanner", "admin")
                if kind == "SemanticModel":
                    self.add_model(row, wsid, "scanner", "admin")
                    self.parse_model_schema(row, wsid)
                if self.config["include_permissions"] and "users" in row:
                    self.add_powerbi_users(row["users"], wsid, iid, kind, "scanner")
                    self.inv.status("scanner_access", iid, "complete" if row["users"] else "empty",
                                    len(row["users"]), "scanner", workspace_id=wsid, item_id=iid)

    def add_report(self, row, wsid, source, scope):
        iid = identifier(row["id"])
        mid = identifier(row.get("datasetId"))
        mws = identifier(row.get("datasetWorkspaceId"))
        self.inv.add("reports", {
            "workspace_id": wsid, "item_id": iid, "report_name": row.get("name"),
            "report_type": row.get("reportType", "Unknown"), "model_id": mid,
            "model_workspace_id": mws, "url": row.get("webUrl"),
        }, source, scope, key=(wsid, iid))
        if mid:
            self.inv.add("lineage", {
                "workspace_id": wsid, "item_id": iid, "source_item_id": mid,
                "source_workspace_id": mws, "target_item_id": iid, "target_workspace_id": wsid,
                "relationship_type": "semantic_model_to_report", "target_name": row.get("name"),
            }, source, scope)

    def add_model(self, row, wsid, source, scope):
        iid = identifier(row["id"])
        self.inv.add("semantic_models", {
            "workspace_id": wsid, "item_id": iid, "model_name": row.get("name"),
            "configured_by": row.get("configuredBy"), "is_refreshable": row.get("isRefreshable"),
            "storage_mode": row.get("targetStorageMode"), "schema_error": safe_text(row.get("schemaRetrievalError")),
        }, source, scope, key=(wsid, iid))

    def parse_model_schema(self, model, wsid):
        iid = identifier(model["id"])
        tables = model.get("tables")
        error = model.get("schemaRetrievalError")
        self.inv.status("model_schema", iid, "unsupported" if error else ("complete" if tables else "unavailable"),
                        len(tables or []), "scanner", workspace_id=wsid, item_id=iid,
                        message=safe_text(error) if error else ("" if tables else "No subartifact schema returned."))
        for table in tables or []:
            name = required(table, "name")
            tid = stable_id(iid, name)
            self.inv.add("tables", {
                "workspace_id": wsid, "item_id": iid, "table_id": tid, "table_name": name,
                "parent_type": "SemanticModel", "table_type": "ModelTable",
                "storage_mode": table.get("storageMode"),
            }, "scanner", "admin", key=(iid, tid))
            for column in table.get("columns") or []:
                self.inv.add("columns", {
                    "workspace_id": wsid, "item_id": iid, "table_id": tid, "table_name": name,
                    "column_name": required(column, "name"), "data_type": column.get("dataType"),
                    "is_hidden": column.get("isHidden"), "is_nullable": column.get("isNullable"),
                }, "scanner", "admin")
            for measure in table.get("measures") or []:
                self.inv.add("measures", {
                    "workspace_id": wsid, "item_id": iid, "table_name": name,
                    "measure_name": required(measure, "name"), "data_type": measure.get("dataType"),
                    "format_string": measure.get("formatString"), "description": measure.get("description"),
                    "is_hidden": measure.get("isHidden"),
                }, "scanner", "admin")
        for rel in model.get("relationships") or []:
            self.inv.add("relationships", {
                "workspace_id": wsid, "item_id": iid, "relationship_name": rel.get("name"),
                "from_table": rel.get("fromTable"), "from_column": rel.get("fromColumn"),
                "to_table": rel.get("toTable"), "to_column": rel.get("toColumn"),
                "is_active": rel.get("isActive"), "cross_filter": rel.get("crossFilteringBehavior"),
            }, "scanner", "admin")
        for source in model.get("upstreamDataflows") or []:
            self.inv.add("lineage", {
                "workspace_id": wsid, "item_id": iid, "source_item_id": identifier(source.get("targetDataflowId")),
                "source_workspace_id": identifier(source.get("groupId")), "target_item_id": iid,
                "target_workspace_id": wsid, "relationship_type": "dataflow_to_model",
                "target_name": model.get("name"),
            }, "scanner", "admin")
        for usage in model.get("datasourceUsages") or []:
            source_id = identifier(usage.get("datasourceInstanceId"))
            source = self.datasource_index.get(source_id, {})
            connection = source.get("connectionDetails") or {}
            self.inv.add("data_sources", {
                "workspace_id": wsid, "item_id": iid, "datasource_id": source_id,
                "datasource_type": source.get("datasourceType"), "server": server_metadata(connection.get("server")),
                "database_name": connection.get("database"), "gateway_id": identifier(source.get("gatewayId")),
            }, "scanner", "admin")

    def supplement_powerbi(self):
        for wsid, ws in self.workspace_map.items():
            if ws.get("state", "Active").lower() != "active":
                continue
            if ws.get("type", "").lower() == "personal":
                self.inv.status("powerbi_supplement", wsid, "not_requested", scope="user",
                                workspace_id=wsid,
                                message="Personal workspace inventory uses admin/scanner metadata; group endpoints do not apply.")
                continue
            for plural, kind in (("reports", "Report"), ("datasets", "SemanticModel"),
                                 ("dashboards", "Dashboard"), ("dataflows", "Dataflow")):
                rows = self.collect_rows("powerbi_" + plural, wsid, self.powerbi,
                                         f"/v1.0/myorg/groups/{wsid}/{plural}", scope="user", workspace_id=wsid)
                for row in rows or []:
                    if kind == "Dataflow" and "id" not in row:
                        row = dict(row, id=required(row, "objectId"))
                    if kind == "Report" and row.get("reportType") == "PaginatedReport":
                        kind_for_item = "PaginatedReport"
                    else:
                        kind_for_item = kind
                    if self.config["item_types"] and kind_for_item not in self.config["item_types"]:
                        continue
                    existing = identifier(row["id"]) in self.item_map
                    if not existing:
                        self.add_item(row, wsid, "powerbi_" + plural, "user", kind_for_item)
                    if kind == "Report":
                        self.add_report(row, wsid, "powerbi_reports", "user")
                    elif kind == "SemanticModel" and not any(r["item_id"] == identifier(row["id"]) for r in self.inv.data("semantic_models")):
                        self.add_model(row, wsid, "powerbi_datasets", "user")

    def add_access(self, row, wsid, iid, kind, source, scope):
        principal = self.add_principal(required(row, "principal"), source, scope)
        details = row.get("workspaceAccessDetails") or row.get("itemAccessDetails") or {}
        entries = []
        if details.get("workspaceRole"):
            entries.append(("role", details["workspaceRole"]))
        entries.extend(("permission", p) for p in details.get("permissions") or [])
        entries.extend(("additional", p) for p in details.get("additionalPermissions") or [])
        ws = self.workspace_map[wsid]
        item = self.item_map.get(iid, {})
        for p_kind, permission in entries:
            self.inv.add("access_assignments", {
                **{k: principal[k] for k in ("principal_id", "principal_name", "principal_type")},
                "capacity_id": identifier(ws.get("capacityId")), "workspace_id": wsid, "item_id": iid,
                "entity_id": iid or wsid, "entity_name": item.get("name", item.get("displayName")) if iid else ws.get("name", ws.get("displayName")),
                "entity_type": kind, "permission": permission, "permission_kind": p_kind,
                "assignment_scope": "item" if iid else "workspace", "inheritance": "unknown",
            }, source, scope)

    def add_powerbi_users(self, users, wsid, iid, kind, source):
        for user in users:
            right = next((v for k, v in user.items() if k.endswith("UserAccessRight")), None)
            if not right:
                continue
            principal = {
                "id": user.get("graphId") or user.get("identifier") or user.get("emailAddress"),
                "displayName": user.get("displayName", user.get("emailAddress")),
                "type": user.get("principalType", "Unknown"),
                "userPrincipalName": user.get("emailAddress"),
            }
            self.add_access({"principal": principal, "itemAccessDetails": {"permissions": [right]}},
                            wsid, iid, kind, source, "admin")

    def permissions(self):
        if not self.config["include_permissions"]:
            self.inv.status("permissions", self.inv.tenant_id, "not_requested")
            return
        for wsid, ws in self.workspace_map.items():
            if ws.get("state", "Active").lower() != "active":
                self.inv.status("workspace_access", wsid, "not_requested", workspace_id=wsid,
                                message="Workspace is not active.")
                continue
            path = f"/v1/admin/workspaces/{wsid}/users"
            users = self.collect_rows("workspace_access", wsid, self.fabric, path, ("accessDetails",),
                                      workspace_id=wsid)
            if users is None:
                fallback = self.collect_rows("workspace_access_user", wsid, self.fabric,
                                             f"/v1/workspaces/{wsid}/roleAssignments", scope="user", workspace_id=wsid)
                for user in fallback or []:
                    self.add_access({"principal": user["principal"], "workspaceAccessDetails": {"workspaceRole": user["role"]}},
                                    wsid, "", "Workspace", "workspace_roleAssignments", "user")
            else:
                for user in users:
                    self.add_access(user, wsid, "", "Workspace", path, "admin")
        scanned = {s["item_id"] for s in self.inv.data("collection_status")
                   if s["collector"] == "scanner_access" and s["status"] in ("complete", "empty")}
        for iid, item in self.item_map.items():
            if iid in scanned:
                continue
            wsid = identifier(item["workspaceId"])
            path = f"/v1/admin/workspaces/{wsid}/items/{iid}/users?" + urlencode({"type": item["type"]})
            users = self.collect_rows("item_access", iid, self.fabric, path, ("accessDetails",),
                                      workspace_id=wsid, item_id=iid)
            for user in users or []:
                self.add_access(user, wsid, iid, item["type"], path, "admin")
        self.expand_groups()

    def expand_groups(self):
        groups = [p for p in self.inv.data("principals") if p["principal_type"] == "Group"]
        for group in groups:
            gid = group["principal_id"]
            if not self.config["expand_groups"]:
                self.inv.status("group_members", gid, "not_requested")
                continue
            if self.graph is None:
                self.inv.status("group_members", gid, "unresolved", scope="user",
                                message="No supported Graph credential is configured in this notebook runtime. Original group grants are retained.")
                continue
            path = f"/v1.0/groups/{gid}/transitiveMembers"
            members = self.collect_rows("group_members", gid, self.graph, path, scope="user")
            for member in members or []:
                self.inv.add("group_memberships", {
                    "group_id": gid, "member_id": identifier(required(member, "id")),
                    "member_name": member.get("displayName"),
                    "member_type": member.get("@odata.type", "Unknown").split(".")[-1],
                    "user_principal_name": member.get("userPrincipalName"), "membership_kind": "transitive",
                }, "graph_transitiveMembers", "user")

    def lakehouse_metadata(self):
        for iid, item in self.item_map.items():
            if item["type"] not in ("Lakehouse", "Warehouse", "SQLDatabase"):
                continue
            wsid = identifier(item["workspaceId"])
            capacity = self.capacity_map.get(identifier(self.workspace_map[wsid].get("capacityId")), {})
            if str(capacity.get("state", "")).lower() in {"suspended", "paused", "inactive"}:
                names = ["item_properties"] + (["lakehouse_tables", "shortcuts"] if item["type"] == "Lakehouse" else [])
                for name in names:
                    self.inv.status(name, iid, "paused", scope="user", workspace_id=wsid, item_id=iid,
                                    message="Source capacity is paused. Workload APIs were not queried; admin catalog metadata is retained.")
                continue
            plural = {"Lakehouse": "lakehouses", "Warehouse": "warehouses", "SQLDatabase": "sqlDatabases"}[item["type"]]
            path = f"/v1/workspaces/{wsid}/{plural}/{iid}"
            try:
                details = self.fabric.get(path, "item_properties")
                props = details.get("properties") or {}
                endpoint = props.get("sqlEndpointProperties") or props
                self.inv.add("endpoints", {
                    "workspace_id": wsid, "item_id": iid, "endpoint_id": identifier(endpoint.get("id")),
                    "endpoint_type": item["type"], "server": server_metadata(endpoint.get("connectionString")),
                    "state": endpoint.get("provisioningStatus"),
                }, path, "user")
                self.inv.status("item_properties", iid, "complete", 1, path, "user", workspace_id=wsid, item_id=iid)
            except ApiFailure as error:
                kind = self.recoverable_failure(error, self.fabric)
                self.inv.status("item_properties", iid, kind, api=path, scope="user",
                                error=error, workspace_id=wsid, item_id=iid)
            if item["type"] != "Lakehouse":
                continue
            path = f"/v1/workspaces/{wsid}/lakehouses/{iid}/tables"
            rows = self.collect_rows("lakehouse_tables", iid, self.fabric, path, ("data",), "user", wsid, iid)
            for table in rows or []:
                name = required(table, "name")
                schema = table.get("schema", "")
                tid = stable_id(iid, schema, name)
                self.inv.add("tables", {
                    "workspace_id": wsid, "item_id": iid, "table_id": tid, "table_name": name,
                    "schema_name": schema, "parent_type": "Lakehouse", "table_type": table.get("type"),
                    "storage_mode": table.get("format"), "location": table.get("location"),
                }, path, "user", key=(iid, tid))
            path = f"/v1/workspaces/{wsid}/items/{iid}/shortcuts"
            shortcuts = self.collect_rows("shortcuts", iid, self.fabric, path, ("value",), "user", wsid, iid)
            for shortcut in shortcuts or []:
                target = shortcut.get("target") or {}
                target_type = next(iter(target), "Unknown")
                onelake = target.get("oneLake") or {}
                self.inv.add("shortcuts", {
                    "workspace_id": wsid, "item_id": iid, "shortcut_name": shortcut.get("name"),
                    "path": shortcut.get("path"), "target_type": target_type,
                    "target_workspace_id": identifier(onelake.get("workspaceId")),
                    "target_item_id": identifier(onelake.get("itemId")), "target_path": onelake.get("path"),
                }, path, "user")
                if onelake.get("itemId"):
                    self.inv.add("lineage", {
                        "workspace_id": wsid, "item_id": iid, "source_item_id": identifier(onelake["itemId"]),
                        "source_workspace_id": identifier(onelake.get("workspaceId")),
                        "target_item_id": iid, "target_workspace_id": wsid,
                        "relationship_type": "onelake_shortcut", "target_name": item.get("name"),
                    }, path, "user")

    def refreshes(self):
        for model in self.inv.data("semantic_models"):
            wsid, iid = model["workspace_id"], model["item_id"]
            if self.workspace_map[wsid].get("type", "").lower() == "personal":
                self.inv.status("refresh_history", iid, "unsupported", scope="user",
                                workspace_id=wsid, item_id=iid,
                                message="Group refresh-history endpoints do not support personal workspaces.")
                continue
            path = f"/v1.0/myorg/groups/{wsid}/datasets/{iid}/refreshes?$top=1"
            rows = self.collect_rows("refresh_history", iid, self.powerbi, path,
                                      scope="user", workspace_id=wsid, item_id=iid)
            for row in rows or []:
                self.inv.add("refresh_status", {
                    "workspace_id": wsid, "item_id": iid, "request_id": row.get("requestId"),
                    "refresh_type": row.get("refreshType"), "state": row.get("status"),
                    "started_at_utc": row.get("startTime"), "ended_at_utc": row.get("endTime"),
                }, path, "user")

    def resolve_lineage(self):
        for edge in self.inv.data("lineage"):
            for side in ("source", "target"):
                item = self.item_map.get(edge[side + "_item_id"])
                if item:
                    edge[side + "_workspace_id"] = identifier(item["workspaceId"])
                    edge[side + "_name"] = item.get("name", item.get("displayName"))

    def activity(self, completed_days=None, event_sink=None):
        if not self.config["include_usage"]:
            self.inv.status("activity", self.inv.tenant_id, "not_requested")
            return
        if self.config["item_types"] and not ({"Report", "PaginatedReport"} & set(self.config["item_types"])):
            self.inv.status("activity", self.inv.tenant_id, "not_requested",
                            message="The collection filter excludes report items.")
            return
        completed_days = set(completed_days or [])
        today = datetime.now(timezone.utc).date()
        selected_ws = set(self.workspace_map)
        for days_back in range(self.config["activity_days"], 0, -1):
            day = today - timedelta(days=days_back)
            ds = day.isoformat()
            if ds in completed_days and days_back > 2:
                continue
            start, end = ds + "T00:00:00.000Z", ds + "T23:59:59.999Z"
            path = "/v1.0/myorg/admin/activityevents?" + urlencode({
                "startDateTime": "'" + start + "'", "endDateTime": "'" + end + "'",
                "$filter": "Activity eq 'ViewReport'",
            })
            count = 0
            try:
                batch = []
                for raw in self.powerbi.pages(path, ("activityEventEntities",), "activity"):
                    wsid = identifier(raw.get("WorkspaceId"))
                    if (self.config["workspace_ids"] or self.config["capacity_ids"]) and wsid not in selected_ws:
                        continue
                    row = normalize_view_event(raw)
                    if row is None:
                        continue
                    stored = self.inv.add("report_view_events", row, "activity_events", "admin",
                                          key=row["event_id"])
                    count += 1
                    if event_sink:
                        batch.append(stored)
                        if len(batch) >= 1000:
                            event_sink(batch)
                            batch.clear()
                            self.inv.rows["report_view_events"].clear()
                if event_sink and batch:
                    event_sink(batch)
                    self.inv.rows["report_view_events"].clear()
                state = "complete"
                self.inv.status("activity", ds, "complete" if count else "empty", count, path)
            except ApiFailure as error:
                state = self.recoverable_failure(error, self.powerbi)
                if error.status == 400 and days_back == 28:
                    state = "outside_retention"
                self.inv.status("activity", ds, state, api=path, error=error)
                if event_sink:
                    self.inv.rows["report_view_events"].clear()
            self.inv.add("activity_day_status", {
                "event_date": ds, "status": state, "event_count": count,
                "window_start_utc": start, "window_end_utc": end,
            }, "activity_events", "admin", key=ds)

    def findings(self):
        counts = Counter(r["workspace_id"] for r in self.inv.data("items"))
        inventory_complete = self.inv.scopes.get("items") == "admin"
        for ws in self.inv.data("workspaces"):
            if not counts[ws["workspace_id"]] and inventory_complete and ws["state"].lower() == "active":
                self.inv.add("findings", {
                    "workspace_id": ws["workspace_id"], "finding_type": "no_items_observed",
                    "summary": "No active items returned for this workspace.",
                    "evidence": "Admin item listing; this is not deletion approval.",
                }, "derived_inventory", "derived")
        for row in self.inv.data("refresh_status"):
            if (row["state"] or "").lower() == "failed":
                self.inv.add("findings", {
                    "workspace_id": row["workspace_id"], "item_id": row["item_id"],
                    "finding_type": "last_refresh_failed", "summary": "Last returned refresh failed.",
                    "evidence": row["started_at_utc"],
                }, "refresh_history", "derived")
        for row in self.inv.data("access_assignments"):
            if row["principal_type"] == "EntireTenant":
                self.inv.add("findings", {
                    "workspace_id": row["workspace_id"], "item_id": row["item_id"],
                    "finding_type": "entire_tenant_assignment", "summary": "An entire-tenant assignment was reported.",
                    "evidence": row["permission"],
                }, row["source_api"], "derived")


def normalize_view_event(raw):
    if str(raw.get("Activity", raw.get("Operation", ""))).lower() != "viewreport":
        return None
    eid = required(raw, "Id")
    timestamp = required(raw, "CreationTime")
    when = datetime.fromisoformat(timestamp.replace("Z", "+00:00"))
    if when.tzinfo is None:
        when = when.replace(tzinfo=timezone.utc)
    when = when.astimezone(timezone.utc)
    actor = identifier(raw.get("UserId") or raw.get("UserKey"))
    user_type = raw.get("UserType")
    # Purview UserType: 0 regular, 2 administrator; other types are not assumed human.
    human = bool(actor) and user_type in (0, 2, "0", "2")
    known_non_human = user_type in (4, 5, 6, 7, 8, "4", "5", "6", "7", "8")
    actor_kind = "human" if human else ("non_human" if known_non_human else "unknown")
    success = raw.get("IsSuccess")
    if success is not None and not isinstance(success, bool):
        success = str(success).lower() == "true"
    return {
        "event_id": identifier(eid), "event_at_utc": when.isoformat(),
        "event_date": when.date().isoformat(), "workspace_id": identifier(raw.get("WorkspaceId")),
        "item_id": identifier(raw.get("ReportId") or raw.get("ArtifactId")),
        "actor_id": actor, "actor_kind": actor_kind, "is_human": human,
        "success": success, "distribution_method": raw.get("DistributionMethod"),
        "consumption_method": raw.get("ConsumptionMethod"),
        "report_name": raw.get("ReportName", raw.get("ItemName")),
        "workspace_name": raw.get("WorkSpaceName", raw.get("WorkspaceName")),
    }


def usage_metrics(events, first_day, last_day, covered_days):
    start, end = date.fromisoformat(first_day), date.fromisoformat(last_day)
    if end < start:
        raise ValueError("Usage end date precedes its start.")
    unique = {}
    for event in events:
        if first_day <= event["event_date"] <= last_day and event.get("success") is not False:
            unique[(event.get("tenant_id", ""), event["event_id"])] = event
    actors = {e["actor_id"] for e in unique.values() if e["is_human"] and e["actor_id"]}
    days = {str(start + timedelta(days=i)) for i in range((end - start).days + 1)}
    coverage = len(days & set(covered_days))
    return {
        "views": len(unique) if coverage else None, "active_users": len(actors) if coverage else None,
        "unknown_actor_events": sum(e["actor_kind"] == "unknown" for e in unique.values()) if coverage else None,
        "requested_days": len(days), "covered_days": coverage,
        "coverage": "complete" if coverage == len(days) else "partial",
    }


def snapshot_changes(before, after, key_fields, compare_fields):
    def index(rows):
        result = {}
        for row in rows:
            key = tuple(row.get(k) for k in key_fields)
            if key in result:
                raise ValueError("Duplicate entity key in snapshot comparison.")
            result[key] = row
        return result
    old, new = index(before), index(after)
    changes = []
    for key in sorted(set(old) | set(new), key=str):
        if key not in old:
            changes.append((key, "newly_observed", "", None, new[key]))
        elif key not in new:
            changes.append((key, "no_longer_observed", "", old[key], None))
        else:
            for field in compare_fields:
                if old[key].get(field) != new[key].get(field):
                    changes.append((key, "changed", field, old[key].get(field), new[key].get(field)))
    return changes


def stable_page(rows, keys, page=1, page_size=100):
    if isinstance(page, bool) or page < 1 or not 1 <= page_size <= 1000:
        raise ValueError("Page must be positive; page size must be between 1 and 1000.")
    ordered = sorted(rows, key=lambda r: tuple(str(r.get(k) or "") for k in keys))
    start = (page - 1) * page_size
    return ordered[start:start + page_size]


In [ ]:
"""Fabric-only adapters. Definitions are safe to load in view-only mode."""

from contextlib import contextmanager


def spark_schema(table):
    from pyspark.sql import types as T
    kinds = {"string": T.StringType(), "boolean": T.BooleanType(), "long": T.LongType(), "date": T.DateType()}
    return T.StructType([T.StructField(name, kinds[kind], True)
                         for name, kind in {**COMMON, **TABLES[table]}.items()])


class SparkInventoryStore:
    def __init__(self, spark_session, config):
        self.spark = spark_session
        self.config = validate_config(config)
        self.workspace_id = identifier(config["destination_workspace_id"])
        self.workspace_name = config["destination_workspace_name"]
        self.lakehouse_id = identifier(config["destination_lakehouse_id"])
        self.lakehouse_name = config["destination_lakehouse_name"]
        if not self.lakehouse_id or not self.workspace_id or not self.lakehouse_name:
            raise ValueError("Attach the dedicated inventory lakehouse and configure its IDs before running.")
        uuid.UUID(self.workspace_id)
        uuid.UUID(self.lakehouse_id)
        for label, value in (("lakehouse", self.lakehouse_name), ("workspace", self.workspace_name)):
            if not isinstance(value, str) or not value.strip() or any(ord(c) < 32 for c in value):
                raise ValueError(f"The {label} display name is missing or contains control characters.")

    def table_name(self, name):
        if name not in TABLES:
            raise ValueError("Only allowlisted inventory tables can be accessed.")
        parts = [self.workspace_name, self.lakehouse_name, "dbo", self.config["table_prefix"] + name]
        return ".".join("`" + part.replace("`", "``") + "`" for part in parts)

    def exists(self, name):
        return self.spark.catalog.tableExists(self.table_name(name))

    def read(self, name):
        if not self.exists(name):
            return self.spark.createDataFrame([], spark_schema(name))
        frame = self.spark.table(self.table_name(name))
        expected = spark_schema(name)
        if [(f.name, f.dataType.simpleString()) for f in frame.schema] != [
                (f.name, f.dataType.simpleString()) for f in expected]:
            raise ValueError(f"Incompatible inventory schema: {name}. No automatic migration was performed.")
        return frame

    def frame(self, name, rows):
        columns = {**COMMON, **TABLES[name]}
        records = [{c: row.get(c) for c in columns} for row in rows]
        return self.spark.createDataFrame(records, spark_schema(name))

    def append(self, name, rows, unique_keys=None):
        if self.config["mode"] != "collect_and_view":
            raise PermissionError("View mode cannot write inventory tables.")
        if not rows and self.exists(name):
            self.read(name)
            return
        frame = self.frame(name, rows)
        keys = unique_keys or ["row_id"]
        frame = frame.dropDuplicates(keys)
        if self.exists(name):
            existing = self.read(name).select(*keys)
            frame = frame.join(existing, keys, "left_anti").select(*spark_schema(name).fieldNames())
            frame.coalesce(1).write.format("delta").mode("append").saveAsTable(self.table_name(name))
        else:
            frame.coalesce(1).write.format("delta").mode("errorifexists").saveAsTable(self.table_name(name))

    def prepare_replacement(self):
        if self.config["write_mode"] != "replace_inventory":
            return
        validate_config(self.config)
        if self.config["mode"] != "collect_and_view" or self.config["confirm_overwrite"] is not True:
            raise PermissionError("Replacement is not confirmed.")
        # This is the only destructive table-write path.
        for name in TABLES:
            if self.exists(name):
                self.frame(name, []).write.format("delta").mode("overwrite").saveAsTable(self.table_name(name))

    def published(self):
        from pyspark.sql import functions as F
        return self.read("published_runs").where(F.col("tenant_id") == self.config["tenant_id"])

    def select_run(self, run_id=None):
        from pyspark.sql import functions as F
        frame = self.published()
        if run_id:
            frame = frame.where(F.col("run_id") == run_id)
        results = frame.orderBy(F.desc("published_at_utc"), F.desc("run_id")).limit(1).collect()
        return results[0].asDict() if results else None

    def snapshot(self, name, run_id):
        from pyspark.sql import functions as F
        return self.read(name).where((F.col("tenant_id") == self.config["tenant_id"]) & (F.col("run_id") == run_id))

    def completed_activity_days(self, fingerprint):
        from pyspark.sql import functions as F
        published = self.published().where(F.col("scope_fingerprint") == fingerprint).select("run_id")
        return {row.event_date for row in self.read("activity_day_status").join(published, "run_id", "inner")
                .where(F.col("status") == "complete").select("event_date").distinct().collect()}

    @contextmanager
    def writer_lock(self, run_id):
        if self.config["mode"] != "collect_and_view":
            raise PermissionError("View mode cannot acquire a writer lock.")
        root = f"abfss://{self.workspace_id}@onelake.dfs.fabric.microsoft.com/{self.lakehouse_id}"
        path = self.spark._jvm.org.apache.hadoop.fs.Path(root + "/Files/inventory_control/writer.lock")
        filesystem = path.getFileSystem(self.spark._jsc.hadoopConfiguration())
        filesystem.mkdirs(path.getParent())
        # ABFS create(overwrite=False) rejects an existing path. Never steal a stale lock.
        stream = filesystem.create(path, False)
        try:
            stream.writeUTF(run_id)
        finally:
            stream.close()
        try:
            yield
        finally:
            reader = filesystem.open(path)
            try:
                owner = reader.readUTF()
            finally:
                reader.close()
            if owner != run_id:
                raise RuntimeError("Writer-lock ownership changed; the lock was not removed.")
            if not filesystem.delete(path, False):
                raise RuntimeError("Could not release the inventory writer lock.")


def assert_notebook_destination(config):
    import notebookutils
    context = notebookutils.runtime.context
    actual = identifier(context.get("defaultLakehouseId"))
    workspace = identifier(context.get("defaultLakehouseWorkspaceId"))
    if actual != identifier(config["destination_lakehouse_id"]):
        raise ValueError("The attached default lakehouse is not the configured inventory destination.")
    if workspace != identifier(config["destination_workspace_id"]):
        raise ValueError("The default lakehouse workspace does not match the configured destination.")
    if context.get("defaultLakehouseName") != config["destination_lakehouse_name"]:
        raise ValueError("The configured lakehouse name does not match the verified lakehouse ID.")
    if context.get("defaultLakehouseWorkspaceName") != config["destination_workspace_name"]:
        raise ValueError("The configured workspace name does not match the verified workspace ID.")
    return context


def make_notebook_apis(config, graph_credential=None):
    import base64
    import importlib.metadata
    import notebookutils
    import requests
    import sempy.fabric as fabric

    token = notebookutils.credentials.getToken("pbi")
    payload = token.split(".")[1]
    claims = json.loads(base64.urlsafe_b64decode(payload + "=" * (-len(payload) % 4)))
    if identifier(claims.get("tid")) != identifier(config["tenant_id"]):
        raise PermissionError("Notebook identity belongs to a different tenant.")
    retry_config = {"total": 0, "raise_on_status": False}
    fabric_client = fabric.FabricRestClient(retry_config=retry_config)
    powerbi_client = fabric.PowerBIRestClient(retry_config=retry_config)

    def transport(client, base):
        def send(method, url, **kwargs):
            relative = url[len(base):]
            try:
                return getattr(client, method.lower())(relative, timeout=60, **kwargs)
            except requests.exceptions.HTTPError as error:
                if error.response is None:
                    raise
                return error.response
        return send

    graph_api = None
    if graph_credential is not None:
        session = requests.Session()

        def graph_transport(method, url, **kwargs):
            access = graph_credential.get_token("https://graph.microsoft.com/.default")
            return session.request(method, url, headers={"Authorization": "Bearer " + access.token},
                                   timeout=60, allow_redirects=False, **kwargs)
        graph_api = RestApi(GRAPH, graph_transport)
    return (
        RestApi(FABRIC, transport(fabric_client, FABRIC)),
        RestApi(POWERBI, transport(powerbi_client, POWERBI)),
        graph_api,
        importlib.metadata.version("semantic-link-sempy"),
    )


def enrich_lakehouse_columns(collector, spark_session):
    from pyspark.errors import AnalysisException
    from py4j.protocol import Py4JJavaError
    import notebookutils

    for iid, item in collector.item_map.items():
        if item["type"] != "Lakehouse" or any(t["item_id"] == iid for t in collector.inv.data("tables")):
            continue
        wsid = identifier(item["workspaceId"])
        capacity = collector.capacity_map.get(identifier(collector.workspace_map[wsid].get("capacityId")), {})
        if str(capacity.get("state", "")).lower() in {"suspended", "paused", "inactive"}:
            collector.inv.status("lakehouse_storage_catalog", iid, "paused", scope="user_storage",
                                 workspace_id=wsid, item_id=iid,
                                 message="Source capacity is paused; OneLake table access is unavailable until it is resumed.")
            continue
        root = f"abfss://{wsid}@onelake.dfs.fabric.microsoft.com/{iid}/Tables"
        discovered = []
        try:
            for entry in notebookutils.fs.ls(root):
                if not entry.isDir:
                    continue
                path = entry.path.rstrip("/")
                if notebookutils.fs.exists(path + "/_delta_log"):
                    discovered.append(("", entry.name.rstrip("/"), path))
                else:
                    for table in notebookutils.fs.ls(path):
                        location = table.path.rstrip("/")
                        if table.isDir and notebookutils.fs.exists(location + "/_delta_log"):
                            discovered.append((entry.name.rstrip("/"), table.name.rstrip("/"), location))
        except (AnalysisException, Py4JJavaError) as error:
            collector.inv.status("lakehouse_storage_catalog", iid, "unavailable", scope="user_storage",
                                 workspace_id=wsid, item_id=iid, message=safe_text(str(error)))
            continue
        for schema_name, table_name, location in discovered:
            tid = stable_id(iid, schema_name, table_name)
            collector.inv.add("tables", {
                "workspace_id": wsid, "item_id": iid, "table_id": tid,
                "table_name": table_name, "schema_name": schema_name,
                "parent_type": "Lakehouse", "table_type": "DeltaTable",
                "storage_mode": "Delta", "location": location,
            }, "onelake_directory_metadata", "user_storage", key=(iid, tid))
        collector.inv.status("lakehouse_storage_catalog", iid, "complete" if discovered else "empty",
                             len(discovered), "onelake_directory_metadata", "user_storage",
                             workspace_id=wsid, item_id=iid)

    for table in collector.inv.data("tables"):
        if table["parent_type"] != "Lakehouse":
            continue
        location = table["location"]
        parsed_location = urlparse(location or "")
        if parsed_location.scheme != "abfss" or parsed_location.hostname != "onelake.dfs.fabric.microsoft.com":
            collector.inv.status("lakehouse_columns", table["table_id"], "unsupported", scope="user",
                                 workspace_id=table["workspace_id"], item_id=table["item_id"],
                                 message="No supported OneLake table location was returned.")
            continue
        try:
            schema = spark_session.read.format("delta").load(location).schema
        except (AnalysisException, Py4JJavaError) as error:
            collector.inv.status("lakehouse_columns", table["table_id"], "unavailable", scope="user",
                                 workspace_id=table["workspace_id"], item_id=table["item_id"],
                                 message=safe_text(str(error)))
            continue
        for field in schema:
            collector.inv.add("columns", {
                "workspace_id": table["workspace_id"], "item_id": table["item_id"],
                "table_id": table["table_id"], "table_name": table["table_name"],
                "column_name": field.name, "data_type": field.dataType.simpleString(),
                "is_nullable": field.nullable,
            }, "delta_schema_metadata", "user")
        collector.inv.status("lakehouse_columns", table["table_id"], "complete", len(schema),
                             "delta_schema_metadata", "user", workspace_id=table["workspace_id"],
                             item_id=table["item_id"])


def enrich_warehouse_catalogs(collector, spark_session):
    from pyspark.errors import AnalysisException
    from py4j.protocol import Py4JJavaError
    for iid, item in collector.item_map.items():
        if item["type"] != "Warehouse":
            continue
        wsid = identifier(item["workspaceId"])
        capacity = collector.capacity_map.get(identifier(collector.workspace_map[wsid].get("capacityId")), {})
        if not capacity or str(capacity.get("state", "")).lower() not in {"active", "suspended", "paused", "inactive"}:
            collector.inv.status("warehouse_catalog", iid, "unverified_capacity", scope="user",
                                 workspace_id=wsid, item_id=iid,
                                 message="No verified active capacity is visible for SQL inspection. Catalog metadata is retained.")
            continue
        if str(capacity.get("state", "")).lower() in {"suspended", "paused", "inactive"}:
            collector.inv.status("warehouse_catalog", iid, "paused", scope="user", workspace_id=wsid, item_id=iid,
                                 message="Source capacity is paused; it was not resumed.")
            continue
        try:
            import com.microsoft.spark.fabric
            from com.microsoft.spark.fabric.Constants import Constants
            reader = (spark_session.read.option(Constants.WorkspaceId, wsid)
                      .option(Constants.DatawarehouseId, iid)
                      .option(Constants.DatabaseName, item.get("name", item.get("displayName"))))
            tables = reader.synapsesql(
                "SELECT TABLE_SCHEMA, TABLE_NAME, TABLE_TYPE FROM INFORMATION_SCHEMA.TABLES"
            ).collect()
            columns = reader.synapsesql(
                "SELECT TABLE_SCHEMA, TABLE_NAME, COLUMN_NAME, DATA_TYPE, IS_NULLABLE FROM INFORMATION_SCHEMA.COLUMNS"
            ).collect()
        except (ImportError, AnalysisException, Py4JJavaError) as error:
            collector.inv.status("warehouse_catalog", iid, "unavailable", scope="user",
                                 workspace_id=wsid, item_id=iid, message=safe_text(str(error)))
            continue
        for table in tables:
            tid = stable_id(iid, table.TABLE_SCHEMA, table.TABLE_NAME)
            collector.inv.add("tables", {
                "workspace_id": wsid, "item_id": iid, "table_id": tid,
                "table_name": table.TABLE_NAME, "schema_name": table.TABLE_SCHEMA,
                "parent_type": "Warehouse", "table_type": table.TABLE_TYPE,
            }, "information_schema", "user", key=(iid, tid))
        for column in columns:
            collector.inv.add("columns", {
                "workspace_id": wsid, "item_id": iid,
                "table_id": stable_id(iid, column.TABLE_SCHEMA, column.TABLE_NAME),
                "table_name": column.TABLE_NAME, "column_name": column.COLUMN_NAME,
                "data_type": column.DATA_TYPE, "is_nullable": column.IS_NULLABLE == "YES",
            }, "information_schema", "user")
        collector.inv.status("warehouse_catalog", iid, "complete", len(tables),
                             "information_schema", "user", workspace_id=wsid, item_id=iid)


def enrich_semantic_model_fallback(collector):
    import requests
    import sempy.fabric as fabric
    for model in collector.inv.data("semantic_models"):
        wsid, iid = model["workspace_id"], model["item_id"]
        if any(t["item_id"] == iid and t["parent_type"] == "SemanticModel" for t in collector.inv.data("tables")):
            continue
        capacity = collector.capacity_map.get(identifier(collector.workspace_map[wsid].get("capacityId")), {})
        if not capacity or str(capacity.get("state", "")).lower() not in {"active", "suspended", "paused", "inactive"}:
            collector.inv.status("model_xmla", iid, "unverified_capacity", scope="user",
                                 workspace_id=wsid, item_id=iid,
                                 message="No verified active capacity is visible for XMLA. Admin/scanner catalog metadata is retained.")
            continue
        if str(capacity.get("state", "")).lower() in {"suspended", "paused", "inactive"}:
            collector.inv.status("model_xmla", iid, "paused", scope="user", workspace_id=wsid, item_id=iid,
                                 message="XMLA fallback skipped on a paused source capacity.")
            continue
        for collection, function in (
            ("tables", fabric.list_tables), ("columns", fabric.list_columns),
            ("measures", fabric.list_measures), ("relationships", fabric.list_relationships),
        ):
            try:
                records = function(dataset=iid, workspace=wsid).to_dict("records")
            except requests.exceptions.HTTPError as error:
                collector.inv.status("model_" + collection, iid, "unavailable", scope="user",
                                     workspace_id=wsid, item_id=iid, message=safe_text(str(error)))
                continue
            for raw in records:
                row = {re.sub(r"[^a-z0-9]", "", key.lower()): value for key, value in raw.items()}
                def value(*names):
                    for name in names:
                        v = row.get(name)
                        if v is not None and str(v) not in ("nan", "<NA>"):
                            return v
                    return None
                name = value("tablename", "table", "name")
                base = {"workspace_id": wsid, "item_id": iid}
                if collection == "tables":
                    if not name:
                        raise ValueError("SemPy list_tables response has no table-name field.")
                    collector.inv.add("tables", {
                        **base, "table_id": stable_id(iid, name), "table_name": name,
                        "parent_type": "SemanticModel", "table_type": "ModelTable",
                    }, "sempy_xmla", "user", key=(iid, stable_id(iid, name)))
                elif collection == "columns":
                    collector.inv.add("columns", {
                        **base, "table_id": stable_id(iid, name), "table_name": name,
                        "column_name": value("columnname", "column", "name"), "data_type": value("datatype", "type"),
                    }, "sempy_xmla", "user")
                elif collection == "measures":
                    collector.inv.add("measures", {
                        **base, "table_name": value("tablename", "table"),
                        "measure_name": value("measurename", "measure", "name"),
                        "data_type": value("datatype"), "format_string": value("formatstring"),
                        "description": value("description"),
                    }, "sempy_xmla", "user")
                else:
                    collector.inv.add("relationships", {
                        **base, "relationship_name": value("relationshipname", "name"),
                        "from_table": value("fromtable"), "from_column": value("fromcolumn"),
                        "to_table": value("totable"), "to_column": value("tocolumn"),
                        "cross_filter": value("crossfilteringbehavior"),
                    }, "sempy_xmla", "user")
            collector.inv.status("model_" + collection, iid, "complete" if records else "empty",
                                 len(records), "sempy_xmla", "user", workspace_id=wsid, item_id=iid)


def append_snapshot_changes(store, inv, fingerprint):
    from pyspark.sql import functions as F
    candidates = (store.published().where(F.col("scope_fingerprint") == fingerprint)
                  .orderBy(F.desc("published_at_utc")).limit(1).collect())
    if not candidates:
        return
    baseline = candidates[0].asDict()
    comparisons = (
        ("workspaces", ["workspace_id"], ["workspace_name", "capacity_id", "description", "state"]),
        ("items", ["workspace_id", "item_id"], ["item_name", "item_type", "capacity_id", "description"]),
    )
    current_status = inv.data("collection_status")
    previous_status = [r.asDict() for r in store.snapshot("collection_status", baseline["run_id"]).collect()]
    for table, keys, fields in comparisons:
        def complete(statuses):
            return any(r["collector"] == table and r["status"] in ("complete", "empty")
                       and r["source_scope"] == "admin" for r in statuses)
        if not complete(current_status) or not complete(previous_status):
            inv.status("comparison_" + table, inv.run_id, "unavailable",
                       message="Comparable complete admin catalog coverage is required.")
            continue
        before = [r.asDict() for r in store.snapshot(table, baseline["run_id"]).collect()]
        for key, kind, field, old, new in snapshot_changes(before, inv.data(table), keys, fields):
            inv.add("changes", {
                "baseline_run_id": baseline["run_id"], "entity_type": table,
                "entity_id": key[-1], "workspace_id": key[0], "item_id": key[-1] if table == "items" else "",
                "change_type": kind, "field_name": field,
                "before_value": json.dumps(old, sort_keys=True) if isinstance(old, dict) else old,
                "after_value": json.dumps(new, sort_keys=True) if isinstance(new, dict) else new,
            }, "snapshot_comparison", "derived")
    access_collectors = {"workspace_access", "workspace_access_user", "item_access", "scanner_access"}
    def covered(statuses):
        return {(s["collector"], s["entity_id"], s["source_scope"]) for s in statuses
                if s["collector"] in access_collectors and s["status"] in ("complete", "empty")}
    comparable = covered(current_status) & covered(previous_status)
    entities = {entity for _, entity, _ in comparable}
    def allowed_assignment(row):
        if row["entity_id"] not in entities:
            return False
        source = row["source_api"]
        if source == "scanner":
            family = "scanner_access"
        elif source == "workspace_roleAssignments":
            family = "workspace_access_user"
        else:
            family = "item_access" if row["item_id"] else "workspace_access"
        return (family, row["entity_id"], row["source_scope"]) in comparable
    before = [r.asDict() for r in store.snapshot("access_assignments", baseline["run_id"]).collect()
              if allowed_assignment(r.asDict())]
    after = [r for r in inv.data("access_assignments") if allowed_assignment(r)]
    keys = ["entity_id", "principal_id", "permission", "permission_kind", "source_api", "source_scope"]
    for key, kind, _, old, new in snapshot_changes(before, after, keys, []):
        row = new or old
        inv.add("changes", {
            "baseline_run_id": baseline["run_id"], "entity_type": "access_assignments",
            "entity_id": row["entity_id"], "workspace_id": row["workspace_id"], "item_id": row["item_id"],
            "change_type": kind, "field_name": row["permission_kind"],
            "before_value": json.dumps(old, sort_keys=True) if old else None,
            "after_value": json.dumps(new, sort_keys=True) if new else None,
        }, "snapshot_comparison", "derived")


def run_collection(store, config, graph_credential=None):
    validate_config(config)
    if config["mode"] != "collect_and_view":
        raise PermissionError("Collection is disabled in view_saved mode.")
    assert_notebook_destination(config)
    fabric_api, powerbi_api, graph_api, version = make_notebook_apis(config, graph_credential)
    inv = Inventory(config["tenant_id"])
    collector = Collector(fabric_api, powerbi_api, config, inv, graph_api)
    with store.writer_lock(inv.run_id):
        store.prepare_replacement()
        started = inv.add("run_events", {
            "event": "started", "status": "running", "message": "Manual inventory collection.",
        }, "inventory_notebook", "local")
        store.append("run_events", [started])
        def step(name, function, *args):
            event = inv.add("run_events", {
                "event": name, "status": "running", "message": "Starting " + name,
            }, "inventory_notebook", "local")
            store.append("run_events", [event])
            try:
                return function(*args)
            except (ValueError, TypeError, ApiFailure) as error:
                failure = inv.add("run_events", {
                    "event": name, "status": "failed",
                    "message": type(error).__name__ + ": " + safe_text(str(error)),
                }, "inventory_notebook", "local")
                store.append("run_events", [failure])
                raise

        step("catalog", collector.catalog)
        fingerprint = scope_fingerprint(config, inv.scopes)
        step("scanner", collector.scanner)
        step("powerbi_metadata", collector.supplement_powerbi)
        if config["include_deep"]:
            step("lakehouse_metadata", collector.lakehouse_metadata)
            step("lakehouse_columns", enrich_lakehouse_columns, collector, store.spark)
            step("warehouse_catalogs", enrich_warehouse_catalogs, collector, store.spark)
            step("model_xmla", enrich_semantic_model_fallback, collector)
            step("refresh_history", collector.refreshes)
        step("permissions", collector.permissions)
        collector.resolve_lineage()
        deferred_tables = {
            "published_runs", "run_events", "report_view_events", "calendar",
            "activity_day_status", "collection_status", "changes", "findings",
        }
        for name in TABLES:
            if name not in deferred_tables:
                store.append(name, inv.data(name))
        previous_days = store.completed_activity_days(fingerprint)
        step("activity", collector.activity, previous_days, lambda rows: store.append(
            "report_view_events", rows, ["tenant_id", "event_id"]))
        collector.findings()
        append_snapshot_changes(store, inv, fingerprint)
        from pyspark.sql import functions as F
        last_date = datetime.now(timezone.utc).date()
        first_date = last_date - timedelta(days=31)
        if store.exists("calendar"):
            saved_first = store.read("calendar").agg(F.min("date").alias("first")).first().first
            if saved_first:
                first_date = min(first_date, saved_first)
        for index in range((last_date - first_date).days + 1):
            day = first_date + timedelta(days=index)
            inv.add("calendar", {
                "date_key": day.isoformat(), "date": day,
                "year": day.year, "month_number": day.month, "month_name": day.strftime("%B"),
            }, "inventory_calendar", "derived", key=day.isoformat())
        for name in ("activity_day_status", "collection_status", "changes", "findings"):
            store.append(name, inv.data(name))
        store.append("calendar", inv.data("calendar"), ["tenant_id", "date_key"])
        if not store.exists("report_view_events"):
            store.append("report_view_events", [])
        gaps = sum(s["status"] not in {"complete", "empty", "not_requested"} for s in inv.data("collection_status"))
        last_day = (datetime.now(timezone.utc).date() - timedelta(days=1)).isoformat()
        publication = inv.add("published_runs", {
            "published_at_utc": utc_now(), "scope_fingerprint": fingerprint,
            "status": "partial" if gaps else "complete", "workspace_count": len(inv.data("workspaces")),
            "item_count": len(inv.data("items")), "coverage_gap_count": gaps,
            "usage_through_date": last_day, "package_version": version,
            "catalog_scope": json.dumps(inv.scopes, sort_keys=True),
        }, "inventory_notebook", "local", key=inv.run_id)
        finished = inv.add("run_events", {
            "event": "validated", "status": publication["status"],
            "message": f"Persisted {len(inv.data('items'))} items. Coverage gaps: {gaps}.",
            "scope_fingerprint": fingerprint,
        }, "inventory_notebook", "local")
        store.append("run_events", [finished])
        store.append("published_runs", [publication])
    return publication


In [ ]:
"""Read-only notebook explorer over published inventory tables."""


class InventoryExplorer:
    VIEWS = (
        "Overview", "Capacities", "Workspaces", "Items", "Deep metadata",
        "Access", "Usage", "Dependencies", "Changes", "Diagnostics", "Collection browser",
    )

    def __init__(self, store, options, display_function):
        self.store = store
        self.spark = store.spark
        self.display = display_function
        self.options = dict(options)
        self.run = None
        self.update(options)

    def update(self, options):
        from datetime import date
        values = dict(options)
        page, size = int(values.get("page", 1)), int(values.get("page_size", 100))
        if page < 1 or not 1 <= size <= 1000:
            raise ValueError("Page must be positive; page_size must be between 1 and 1000.")
        for key in ("capacity_id", "workspace_id", "item_id", "principal_id"):
            values[key] = identifier(values.get(key))
        values["page"], values["page_size"] = page, size
        for field in ("start_date", "end_date"):
            if values.get(field):
                date.fromisoformat(values[field])
        if values.get("start_date") and values.get("end_date") and values["start_date"] > values["end_date"]:
            raise ValueError("The usage date range is reversed.")
        self.run = self.store.select_run(values.get("run_id") or None)
        if values.get("run_id") and self.run is None:
            raise ValueError("The selected run is not published or does not belong to this inventory tenant.")
        self.options = values

    def title(self, text):
        from IPython.display import Markdown, display as ipython_display
        ipython_display(Markdown("## " + text))
        if self.run:
            print(f"Snapshot: {self.run['run_id']} | Published: {self.run['published_at_utc']}")
            print(f"Catalog scope: {self.run['catalog_scope']} | Result: {self.run['status']} | Coverage gaps: {self.run['coverage_gap_count']}")
        print("View controls:", json.dumps({k: v for k, v in self.options.items() if v not in (None, "", [], False)}, sort_keys=True))
        print("Rendered:", utc_now(), "| Views do not collect or write data.")

    def frame(self, name, filtered=True):
        from pyspark.sql import functions as F
        if self.run is None:
            return self.store.frame(name, [])
        if not self.store.exists(name):
            raise ValueError(f"Published inventory is missing table {name}; inspect diagnostics.")
        if name == "published_runs":
            frame = self.store.published()
        elif name == "calendar":
            frame = self.store.read(name).where(F.col("tenant_id") == self.store.config["tenant_id"])
        elif name == "report_view_events":
            frame = self.events()
        elif name == "activity_day_status":
            frame = self.day_status()
        else:
            frame = self.store.snapshot(name, self.run["run_id"])
        if not filtered:
            return frame
        for key in ("capacity_id", "workspace_id", "item_id"):
            value = self.options.get(key)
            if value and key in frame.columns:
                frame = frame.where(F.col(key) == value)
            elif value and key == "capacity_id" and "workspace_id" in frame.columns:
                ws = self.store.snapshot("workspaces", self.run["run_id"]).where(
                    F.col("capacity_id") == value).select("workspace_id")
                frame = frame.join(ws, "workspace_id", "left_semi")
            elif value and key == "workspace_id" and name == "capacities":
                ancestors = self.store.snapshot("workspaces", self.run["run_id"]).where(
                    F.col("workspace_id") == value).select("capacity_id")
                frame = frame.join(ancestors, "capacity_id", "left_semi")
            elif value and key == "item_id" and name in ("capacities", "workspaces"):
                ancestor_key = "capacity_id" if name == "capacities" else "workspace_id"
                ancestors = self.store.snapshot("items", self.run["run_id"]).where(
                    F.col("item_id") == value).select(ancestor_key)
                frame = frame.join(ancestors, ancestor_key, "left_semi")
        if self.options.get("item_type"):
            if "item_type" in frame.columns:
                frame = frame.where(F.col("item_type") == self.options["item_type"])
            elif "item_id" in frame.columns:
                items = self.store.snapshot("items", self.run["run_id"]).where(
                    F.col("item_type") == self.options["item_type"]).select("item_id")
                frame = frame.join(items, "item_id", "left_semi")
        if self.options.get("domain_id") and "workspace_id" in frame.columns:
            ws = self.store.snapshot("workspaces", self.run["run_id"]).where(
                F.col("domain_id") == self.options["domain_id"]).select("workspace_id")
            frame = frame.join(ws, "workspace_id", "left_semi")
        if self.options.get("tag") and "workspace_id" in frame.columns:
            tags = self.store.snapshot("tags", self.run["run_id"]).where(
                F.lower(F.col("tag_name")) == self.options["tag"].lower())
            ws_tags = tags.where(F.col("entity_type") == "Workspace").select("workspace_id")
            if "item_id" in frame.columns:
                tagged_items = tags.select(F.col("entity_id").alias("item_id")).distinct()
                tagged_ws_items = frame.join(ws_tags, "workspace_id", "left_semi").select("row_id")
                own_items = frame.join(tagged_items, "item_id", "left_semi").select("row_id")
                frame = frame.join(tagged_ws_items.union(own_items).distinct(), "row_id", "left_semi")
            else:
                frame = frame.join(ws_tags, "workspace_id", "left_semi")
        if self.options.get("state") and "state" in frame.columns:
            frame = frame.where(F.lower(F.col("state")) == self.options["state"].lower())
        text = self.options.get("search", "").strip().lower()
        if text:
            names = [c for c in frame.columns if c.endswith("_name") or c in {"description", "summary", "message"}]
            if names:
                condition = F.lit(False)
                for name in names:
                    condition = condition | F.lower(F.coalesce(F.col(name), F.lit(""))).contains(text)
                frame = frame.where(condition)
        return frame

    def page(self, frame, label, columns=None, page=None):
        from pyspark.sql import functions as F, Window
        total = frame.count()
        number = page or self.options["page"]
        size = self.options["page_size"]
        first = (number - 1) * size
        print(f"{label}: {total:,} matching rows | page {number}/{max(1, (total + size - 1) // size)} | page size {size}")
        if not total:
            print("No matching rows. Consult Diagnostics to distinguish an empty collection from unavailable data.")
            return frame
        if first >= total:
            print("This page is beyond the result. Set page=1 or select an earlier page.")
            return frame
        names = [c for c in frame.columns if c.endswith("_name")]
        keys = names + (["row_id"] if "row_id" in frame.columns else sorted(set(frame.columns) - set(names)))
        ordering = [F.col(c).asc_nulls_last() for c in keys]
        paged = frame.withColumn("__page_row", F.row_number().over(Window.orderBy(*ordering)))
        paged = paged.where((F.col("__page_row") > first) & (F.col("__page_row") <= first + size)).drop("__page_row")
        if columns:
            paged = paged.select(*[c for c in columns if c in paged.columns])
        self.display(paged)
        return frame

    def chart(self, frame, category, value, title, kind="bar", limit=20):
        import matplotlib.pyplot as plt
        from pyspark.sql import functions as F
        ordered = frame.orderBy(F.col(category)) if kind == "line" else frame.orderBy(F.desc(value), category)
        data = ordered.select(category, value).limit(limit).toPandas()
        if data.empty:
            print(title + ": no chart data.")
            return
        fig, ax = plt.subplots(figsize=(10, 3.5))
        if kind == "line":
            ax.plot(data[category].astype(str), data[value], color="#28664f", marker=".")
            ax.tick_params(axis="x", rotation=55)
        else:
            ax.barh(data[category].fillna("(unknown)").astype(str), data[value], color="#28664f")
            ax.invert_yaxis()
        ax.set_title(title + (f" (top {limit}; complete table available)" if kind == "bar" else ""))
        ax.set_xlabel(value.replace("_", " "))
        fig.tight_layout()
        plt.show()
        plt.close(fig)

    def contextual_items(self):
        from pyspark.sql import functions as F
        workspaces = self.frame("workspaces", False).select("workspace_id", "workspace_name", "domain_id")
        capacities = self.frame("capacities", False).select("capacity_id", "capacity_name", "sku", "region")
        return self.frame("items").join(workspaces, "workspace_id", "left").join(capacities, "capacity_id", "left")

    def overview(self):
        from pyspark.sql import functions as F
        self.title("Inventory overview and coverage")
        values = [("Capacities observed", self.frame("capacities").count()),
                  ("Workspaces observed", self.frame("workspaces").count()),
                  ("Items observed", self.frame("items").count()),
                  ("Collection gaps", self.frame("collection_status", False).where(
                      ~F.col("status").isin("complete", "empty", "not_requested")).count())]
        self.display(self.spark.createDataFrame(values, "measure string, value long"))
        counts = self.frame("items").groupBy("item_type").count().withColumnRenamed("count", "items")
        self.chart(counts, "item_type", "items", "Observed item types")
        self.page(counts, "All item types")
        status = self.frame("collection_status", False).groupBy("collector", "source_scope", "status").count()
        self.page(status, "Collection coverage")

    def capacities(self):
        from pyspark.sql import functions as F
        self.title("Capacities")
        ws = self.frame("workspaces", False).groupBy("capacity_id").count().withColumnRenamed("count", "workspace_count")
        items = self.frame("items", False).groupBy("capacity_id").count().withColumnRenamed("count", "item_count")
        result = self.frame("capacities").join(ws, "capacity_id", "left").join(items, "capacity_id", "left")
        self.page(result.fillna(0, ["workspace_count", "item_count"]), "All returned capacities")
        self.page(self.frame("workspaces", False).join(
            self.frame("capacities", False).select("capacity_id"), "capacity_id", "left_anti"),
            "Workspaces with shared, missing, or unavailable capacity metadata")

    def workspaces(self):
        self.title("Workspaces by capacity")
        counts = self.frame("items", False).groupBy("workspace_id").count().withColumnRenamed("count", "item_count")
        capacities = self.frame("capacities", False).select("capacity_id", "capacity_name", "sku", "region")
        domains = self.frame("domains", False).select("domain_id", "domain_name")
        frame = self.frame("workspaces").join(counts, "workspace_id", "left").join(
            capacities, "capacity_id", "left").join(domains, "domain_id", "left").fillna(0, ["item_count"])
        self.page(frame, "Workspaces")
        self.page(self.frame("items").groupBy("workspace_id", "item_type").count(), "Workspace / item-type counts")

    def items(self):
        self.title("Items and selected-item details")
        self.page(self.contextual_items(), "All item types and available metadata")
        if self.options.get("item_id"):
            self.record("items", item_id=self.options["item_id"])
            self.page(self.frame("tags"), "Item/workspace tags")
            self.page(self.frame("endpoints"), "Recorded endpoints")
            self.page(self.frame("refresh_status"), "Last recorded refresh")

    def deep_metadata(self):
        self.title("Deep metadata")
        print("These are schemas and model metadata, not business-table contents. Paused or denied sources remain visible in Diagnostics.")
        name = self.options.get("collection", "tables")
        allowed = {"semantic_models", "reports", "tables", "columns", "measures", "relationships", "data_sources", "endpoints", "shortcuts", "refresh_status"}
        if name not in allowed:
            name = "tables"
        self.page(self.frame(name), name)
        print("Choose another deep collection in the controls. Every collection is also in Collection browser.")

    def access_paths(self):
        from pyspark.sql import functions as F
        raw = self.frame("access_assignments")
        direct = (raw.withColumn("subject_id", F.col("principal_id"))
                  .withColumn("subject_name", F.col("principal_name"))
                  .withColumn("subject_type", F.col("principal_type"))
                  .withColumn("granted_group_id", F.lit(""))
                  .withColumn("access_path", F.lit("reported_assignment")))
        members = self.frame("group_memberships", False)
        through = raw.where(F.col("principal_type") == "Group").join(
            members.select("group_id", "member_id", "member_name", "member_type"),
            raw.principal_id == members.group_id, "inner")
        through = (through.withColumn("subject_id", F.col("member_id"))
                   .withColumn("subject_name", F.col("member_name"))
                   .withColumn("subject_type", F.col("member_type"))
                   .withColumn("granted_group_id", F.col("group_id"))
                   .withColumn("access_path", F.lit("through_group"))
                   .withColumn("row_id", F.sha2(F.concat_ws("|", "row_id", "member_id"), 256))
                   .select(*direct.columns))
        result = direct.unionByName(through)
        if self.options.get("principal_id"):
            result = result.where((F.col("subject_id") == self.options["principal_id"])
                                  | (F.col("principal_id") == self.options["principal_id"]))
        if self.options.get("permission"):
            result = result.where(F.lower(F.col("permission")) == self.options["permission"].lower())
        return result

    def access(self):
        self.title("Access explorer")
        print("Observed assignments and group-derived paths. Not a complete RLS, OLS, SQL, or OneLake authorization decision.")
        paths = self.access_paths()
        self.page(paths, "Recorded access paths")
        print("Distinct subjects in this view:", paths.select("subject_id").distinct().count(),
              "| Path rows are not a unique-person count.")
        from pyspark.sql import functions as F
        unresolved = self.frame("collection_status", False).where(
            (F.col("collector") == "group_members") & ~F.col("status").isin("complete", "empty"))
        self.page(unresolved, "Unresolved or unrequested group membership")

    def day_status(self):
        from pyspark.sql import functions as F, Window
        if self.run is None:
            return self.store.frame("activity_day_status", [])
        pubs = (self.store.published().where(
            (F.col("scope_fingerprint") == self.run["scope_fingerprint"])
            & (F.col("published_at_utc") <= self.run["published_at_utc"]))
                .select("run_id", "published_at_utc"))
        status = self.store.read("activity_day_status").join(pubs, "run_id", "inner")
        window = Window.partitionBy("event_date").orderBy(F.desc("published_at_utc"), F.desc("run_id"))
        return (status.withColumn("__latest", F.row_number().over(window))
                .where(F.col("__latest") == 1).drop("__latest", "published_at_utc"))

    def events(self):
        from pyspark.sql import functions as F
        if self.run is None:
            return self.store.frame("report_view_events", [])
        days = self.day_status().where(F.col("status") == "complete").select("event_date")
        frame = self.store.read("report_view_events").where(
            (F.col("tenant_id") == self.store.config["tenant_id"])
            & (F.col("collected_at_utc") <= self.run["published_at_utc"])
            & (F.col("success").isNull() | F.col("success")))
        return frame.join(days, "event_date", "inner").dropDuplicates(["tenant_id", "event_id"])

    def usage_window(self):
        end = self.options.get("end_date") or self.run["usage_through_date"]
        start = self.options.get("start_date") or (date.fromisoformat(end) - timedelta(days=27)).isoformat()
        if start > end:
            raise ValueError("Usage window is reversed.")
        return start, end

    def usage_summary(self, events, start, end):
        from pyspark.sql import functions as F
        days = (date.fromisoformat(end) - date.fromisoformat(start)).days + 1
        covered = self.day_status().where(
            (F.col("event_date") >= start) & (F.col("event_date") <= end)
            & (F.col("status") == "complete")).count()
        selected = events.where((F.col("event_date") >= start) & (F.col("event_date") <= end))
        row = selected.agg(
            F.count("*").alias("views"),
            F.countDistinct(F.when(F.col("is_human") & (F.col("actor_id") != ""), F.col("actor_id"))).alias("active_users"),
            F.count(F.when(F.col("actor_kind") == "unknown", 1)).alias("unknown_actor_events"),
        ).first().asDict()
        if not covered:
            row = {key: None for key in row}
        return dict(row, start_date=start, end_date=end, requested_days=days,
                    covered_days=covered, coverage="complete" if covered == days else "partial")

    def usage(self):
        from pyspark.sql import functions as F
        self.title("Report adoption and observation coverage")
        start, end = self.usage_window()
        events = self.frame("report_view_events")
        windows = [
            ("Selected period", start, end),
            ("WAU", (date.fromisoformat(end) - timedelta(days=6)).isoformat(), end),
            ("MAU-30", (date.fromisoformat(end) - timedelta(days=29)).isoformat(), end),
            ("Calendar month through selected end", date.fromisoformat(end).replace(day=1).isoformat(), end),
        ]
        values = [dict(metric=label, **self.usage_summary(events, a, b)) for label, a, b in windows]
        import pandas as pd
        self.display(pd.DataFrame(values))
        print("Distinct users are calculated across the window, never by summing daily or per-report unique users.")
        print("Unknown or automated actors are excluded from human MAU. Longer windows may be partial.")
        selected = events.where((F.col("event_date") >= start) & (F.col("event_date") <= end))
        daily = selected.groupBy("event_date").agg(
            F.count("*").alias("views"),
            F.countDistinct(F.when(F.col("is_human"), F.col("actor_id"))).alias("active_users"))
        self.chart(daily, "event_date", "views", "Recorded report views by UTC day", kind="line", limit=366)
        self.page(daily, "Daily views and active users")
        ranked = selected.groupBy("workspace_id", "item_id").agg(
            F.count("*").alias("views"),
            F.countDistinct(F.when(F.col("is_human"), F.col("actor_id"))).alias("active_users"),
            F.max("event_at_utc").alias("last_observed_view"))
        reports = self.frame("reports").select("workspace_id", "item_id", "report_name")
        ranked = reports.join(ranked, ["workspace_id", "item_id"], "full_outer")
        if values[0]["covered_days"] > 0:
            ranked = ranked.fillna(0, ["views", "active_users"])
        self.page(ranked, "All report rankings, including no observed views and unresolved report IDs")
        self.page(self.day_status(), "Per-day extraction coverage")

    def dependencies(self):
        from pyspark.sql import functions as F
        self.title("Known dependencies")
        edges = self.frame("lineage", False)
        if self.options.get("item_id"):
            iid = self.options["item_id"]
            self.page(edges.where(F.col("target_item_id") == iid), "Upstream of selected item")
            self.page(edges.where(F.col("source_item_id") == iid), "Downstream of selected item")
        else:
            self.page(self.frame("lineage"), "All known dependency edges")
        print("No edge means no recorded dependency evidence, not proof of independence.")

    def changes(self):
        from pyspark.sql import functions as F
        self.title("Changes and attention")
        self.page(self.frame("findings"), "Evidence-backed attention list")
        baseline_id = self.options.get("baseline_run_id")
        if not baseline_id:
            self.page(self.frame("changes"), "Changes from the prior comparable published run")
            print("Select baseline_run_id to compare a different published snapshot.")
            return
        baseline = self.store.select_run(baseline_id)
        if baseline is None or baseline["scope_fingerprint"] != self.run["scope_fingerprint"]:
            print("Comparison unavailable: the baseline is unpublished or uses a different collection scope.")
            return
        for name, keys, fields in (
            ("workspaces", ["workspace_id"], ["workspace_name", "capacity_id", "description", "state"]),
            ("items", ["workspace_id", "item_id"], ["item_name", "item_type", "capacity_id", "description"]),
        ):
            def complete(run_id):
                return self.store.snapshot("collection_status", run_id).where(
                    (F.col("collector") == name) & F.col("status").isin("complete", "empty")
                    & (F.col("source_scope") == "admin")).count() > 0
            if not complete(baseline_id) or not complete(self.run["run_id"]):
                print(name + ": comparison unavailable because catalog coverage is not comparable.")
                continue
            before = self.store.snapshot(name, baseline_id).select(*keys, F.to_json(F.struct(*fields)).alias("before"))
            after = self.store.snapshot(name, self.run["run_id"]).select(*keys, F.to_json(F.struct(*fields)).alias("after"))
            changed = before.join(after, keys, "full_outer").where(~F.col("before").eqNullSafe(F.col("after")))
            changed = changed.withColumn("change_type", F.when(F.col("before").isNull(), "newly_observed")
                                         .when(F.col("after").isNull(), "no_longer_observed").otherwise("changed"))
            self.page(changed, name + " before/after")
        print("Snapshots do not establish the exact change time, change actor, or confirmed deletion.")

    def diagnostics(self):
        from pyspark.sql import functions as F
        self.title("Collection diagnostics")
        frame = self.frame("collection_status")
        hints = F.lit("")
        for state in ("failed", "unsupported", "unverified_capacity", "denied", "unresolved", "outside_retention", "throttled", "paused"):
            hints = F.when(F.col("status") == state, F.lit(collection_guidance("", state, "user"))).otherwise(hints)
        hints = F.when(
            (F.col("status") == "denied") & (F.col("source_scope") == "admin"),
            F.lit(collection_guidance("", "denied", "admin")),
        ).otherwise(hints)
        hints = F.when(
            (F.col("status") == "denied") & F.col("collector").startswith("model_"),
            F.lit(collection_guidance("model_metadata", "denied", "user")),
        ).otherwise(hints)
        hints = F.when(
            (F.col("status") == "denied")
            & F.lower(F.concat_ws(" ", F.col("error_code"), F.col("message"))).contains("scope"),
            F.lit(collection_guidance("", "denied", "user", "missing API scope")),
        ).otherwise(hints)
        self.page(frame.withColumn("next_step", hints), "Selected-run status and recovery guidance")
        self.page(self.store.published(), "Published runs and observation boundaries")
        published_ids = self.store.published().select("run_id").distinct()
        self.page(self.store.read("run_events").join(published_ids, "run_id", "left_anti"),
                  "Unpublished-run diagnostics (not included in normal inventory totals)")
        print("A stale writer lock is never removed automatically. Confirm that the previous writer stopped before clearing its exact lock file.")

    def collection_browser(self):
        self.title("Saved collection browser")
        import pandas as pd
        entries = []
        for name in TABLES:
            entries.append({"collection": name, "fields": ", ".join(TABLES[name]),
                            "persisted": self.store.exists(name)})
        self.display(pd.DataFrame(entries))
        name = self.options.get("collection") or "items"
        if name not in TABLES:
            raise ValueError("Collection name is not allowlisted.")
        self.page(self.frame(name), "Complete saved collection: " + name)
        print("All columns are available. Use explorer.record(collection, row_id=...) for a vertical record detail.")

    def record(self, collection, **criteria):
        from pyspark.sql import functions as F
        if not criteria:
            raise ValueError("Specify row_id or an entity ID to inspect a record.")
        frame = self.frame(collection, False)
        for key, value in criteria.items():
            if key not in frame.columns:
                raise ValueError("Unknown record field: " + key)
            frame = frame.where(F.col(key) == value)
        rows = frame.limit(2).collect()
        if not rows:
            print("No matching record in the selected published run.")
            return
        if len(rows) > 1:
            raise ValueError("The selection is not unique. Use row_id.")
        import pandas as pd
        self.display(pd.DataFrame(rows[0].asDict().items(), columns=["field", "value"]))

    def render(self, view=None):
        if self.run is None:
            print("No published inventory exists. Set MODE='collect_and_view' explicitly, then run the collection cell.")
            print("No source APIs were called and no tables were created.")
            return
        view = view or self.options.get("view", "Overview")
        methods = {
            "Overview": self.overview, "Capacities": self.capacities, "Workspaces": self.workspaces,
            "Items": self.items, "Deep metadata": self.deep_metadata, "Access": self.access,
            "Usage": self.usage, "Dependencies": self.dependencies, "Changes": self.changes,
            "Diagnostics": self.diagnostics, "Collection browser": self.collection_browser,
        }
        if view not in methods:
            raise ValueError("Unknown notebook view.")
        methods[view]()

    def controls(self):
        """One output area ensures a changed filter cannot leave unmarked stale views."""
        try:
            import ipywidgets as widgets
        except ImportError:
            print("Widget controls are unavailable. Edit VIEW, then run explorer.update(VIEW); explorer.render().")
            self.render()
            return
        from IPython.display import display as ipython_display
        output = widgets.Output()
        view = widgets.Dropdown(options=self.VIEWS, value=self.options.get("view", "Overview"), description="View")
        collection = widgets.Dropdown(options=list(TABLES), value=self.options.get("collection", "items"), description="Collection")
        fields = {}
        for key, label in (
            ("run_id", "Snapshot"), ("baseline_run_id", "Compare"),
            ("capacity_id", "Capacity ID"), ("workspace_id", "Workspace ID"),
            ("item_id", "Item ID"), ("item_type", "Item type"),
            ("principal_id", "Principal ID"), ("permission", "Permission"),
            ("domain_id", "Domain ID"), ("tag", "Tag"), ("state", "State"),
            ("search", "Search"), ("start_date", "Start UTC"), ("end_date", "End UTC"),
        ):
            fields[key] = widgets.Text(value=self.options.get(key) or "", description=label,
                                       layout=widgets.Layout(width="48%"))
        page = widgets.BoundedIntText(value=self.options["page"], min=1, max=1000000, description="Page")
        size = widgets.BoundedIntText(value=self.options["page_size"], min=1, max=1000, description="Rows")
        apply = widgets.Button(description="Apply view", button_style="primary")
        reset = widgets.Button(description="Clear filters")
        previous = widgets.Button(description="Previous page")
        following = widgets.Button(description="Next page")

        def show(_=None):
            output.clear_output(wait=True)
            with output:
                opts = {key: field.value.strip() for key, field in fields.items()}
                opts.update(view=view.value, collection=collection.value, page=page.value, page_size=size.value)
                try:
                    self.update(opts)
                    self.render()
                except (ValueError, PermissionError) as error:
                    print("View was not rendered:", error)
        def clear(_):
            for field in fields.values():
                field.value = ""
            page.value = 1
            show()
        def move_previous(_):
            page.value = max(1, page.value - 1)
            show()
        def move_next(_):
            page.value += 1
            show()
        apply.on_click(show)
        reset.on_click(clear)
        previous.on_click(move_previous)
        following.on_click(move_next)
        ipython_display(widgets.VBox([
            widgets.HTML("<b>Read-only saved inventory</b>. Select a view, enter filters, then Apply view. Blank snapshot means latest published."),
            widgets.HBox([view, collection]),
            widgets.Box(list(fields.values()), layout=widgets.Layout(display="flex", flex_flow="row wrap")),
            widgets.HBox([page, size, apply, reset, previous, following]), output,
        ]))
        show()


In [ ]:
"""Original TMDL and PBIR definitions for the inventory companion report."""
import argparse
import base64
import hashlib
import json
import re
import uuid
from pathlib import Path
SCHEMA_ROOT = 'https://developer.microsoft.com/json-schemas/fabric/item/report/'
ALIASES = {'published_runs': 'Runs', 'capacities': 'Capacities', 'workspaces': 'Workspaces', 'items': 'Items', 'reports': 'Reports', 'semantic_models': 'Semantic models', 'tables': 'Tables', 'columns': 'Columns', 'measures': 'Model measures', 'relationships': 'Relationships', 'data_sources': 'Data sources', 'access_assignments': 'Access', 'group_memberships': 'Group members', 'principals': 'Principals', 'report_view_events': 'Views', 'activity_day_status': 'Activity days', 'collection_status': 'Collection status', 'changes': 'Changes', 'findings': 'Findings', 'lineage': 'Lineage', 'domains': 'Domains', 'tags': 'Tags', 'refresh_status': 'Refresh status', 'endpoints': 'Endpoints', 'shortcuts': 'Shortcuts', 'calendar': 'Date'}

def tmdl_name(name):
    return "'" + name.replace("'", "''") + "'"

def object_name(*parts):
    return hashlib.sha256('|'.join(parts).encode()).hexdigest()[:20]

def formula_measures():
    measures = {'Selected run': ("VAR _chosen = SELECTEDVALUE('Runs'[run_id])\nVAR _latest = MAXX(TOPN(1, ALL('Runs'), 'Runs'[published_at_utc], DESC, 'Runs'[run_id], DESC), 'Runs'[run_id])\nRETURN COALESCE(_chosen, _latest)", ''), 'Snapshot timestamp': ("VAR _run = [Selected run]\nRETURN LOOKUPVALUE('Runs'[published_at_utc], 'Runs'[run_id], _run)", ''), 'Snapshot status': ("VAR _run = [Selected run]\nRETURN LOOKUPVALUE('Runs'[status], 'Runs'[run_id], _run)", ''), 'Collection gaps': ("VAR _run = [Selected run]\nRETURN LOOKUPVALUE('Runs'[coverage_gap_count], 'Runs'[run_id], _run)", '#,##0'), 'Window end': ("VAR _run = [Selected run]\nVAR _through = DATEVALUE(LOOKUPVALUE('Runs'[usage_through_date], 'Runs'[run_id], _run))\nRETURN IF(ISFILTERED('Date'[date]), MIN(MAX('Date'[date]), _through), _through)", 'yyyy-MM-dd'), 'Window start': ("IF(ISFILTERED('Date'[date]), MIN('Date'[date]), [Window end] - 27)", 'yyyy-MM-dd'), 'Day complete': ('VAR _run = [Selected run]\nVAR _day = SELECTEDVALUE(\'Date\'[date_key])\nVAR _stamp = [Snapshot timestamp]\nVAR _scope = LOOKUPVALUE(\'Runs\'[scope_fingerprint], \'Runs\'[run_id], _run)\nVAR _runs = SELECTCOLUMNS(FILTER(ALL(\'Runs\'), \'Runs\'[scope_fingerprint] = _scope && \'Runs\'[published_at_utc] <= _stamp), "rid", \'Runs\'[run_id])\nVAR _candidates = ADDCOLUMNS(FILTER(ALL(\'Activity days\'), \'Activity days\'[event_date] = _day && \'Activity days\'[run_id] IN _runs), "pub", LOOKUPVALUE(\'Runs\'[published_at_utc], \'Runs\'[run_id], \'Activity days\'[run_id]))\nVAR _last = TOPN(1, _candidates, [pub], DESC, \'Activity days\'[run_id], DESC)\nRETURN IF(MAXX(_last, \'Activity days\'[status]) = "complete", 1, 0)', '0'), 'Covered days': ("VAR _first = [Window start]\nVAR _last = [Window end]\nRETURN SUMX(FILTER(ALL('Date'), 'Date'[date] >= _first && 'Date'[date] <= _last), [Day complete])", '#,##0'), 'Requested days': ('MAX(0, INT([Window end] - [Window start]) + 1)', '#,##0'), 'Usage coverage': ('FORMAT([Covered days], "0") & " / " & FORMAT([Requested days], "0") & " days; " & IF([Covered days] = [Requested days], "complete", "partial")', '')}
    for key, alias in ALIASES.items():
        if key in {'published_runs', 'report_view_events', 'activity_day_status', 'calendar'}:
            continue
        measures['Rows ' + alias] = (f'VAR _run = [Selected run]\nRETURN CALCULATE(COUNTROWS({tmdl_name(alias)}), KEEPFILTERS({tmdl_name(alias)}[run_id] = _run))', '#,##0')
    expressions = {'Report views': "DISTINCTCOUNT('Views'[event_id])", 'Active users': 'CALCULATE(DISTINCTCOUNT(\'Views\'[actor_id]), \'Views\'[is_human] = TRUE(), \'Views\'[actor_id] <> "")', 'Unknown actor views': 'CALCULATE(DISTINCTCOUNT(\'Views\'[event_id]), \'Views\'[actor_kind] = "unknown")', 'Last observed view': "MAX('Views'[event_at_utc])"}
    for name, aggregation in expressions.items():
        result = "IF(ISCROSSFILTERED('Items'), _filtered, _all)"
        if name != 'Last observed view':
            result = f'COALESCE({result}, 0)'
        measures[name] = (f"""VAR _run = [Selected run]\nVAR _stamp = [Snapshot timestamp]\nVAR _first = [Window start]\nVAR _last = [Window end]\nVAR _dates = SELECTCOLUMNS(FILTER(ALL('Date'), 'Date'[date] >= _first && 'Date'[date] <= _last && [Day complete] = 1), "day", 'Date'[date_key])\nVAR _items = CALCULATETABLE(VALUES('Items'[item_id]), KEEPFILTERS('Items'[run_id] = _run))\nVAR _events = FILTER('Views', 'Views'[collected_at_utc] <= _stamp && ('Views'[success] = TRUE() || ISBLANK('Views'[success])))\nVAR _all = CALCULATE({aggregation}, _events, KEEPFILTERS(TREATAS(_dates, 'Views'[event_date])))\nVAR _filtered = CALCULATE({aggregation}, _events, KEEPFILTERS(TREATAS(_dates, 'Views'[event_date])), KEEPFILTERS(TREATAS(_items, 'Views'[item_id])))\nRETURN IF([Covered days] = 0, BLANK(), {result})""", '' if name == 'Last observed view' else '#,##0')
    for name, start_expression in (('WAU', '_last - 6'), ('MAU-30', '_last - 29'), ('Calendar month active users', 'DATE(YEAR(_last), MONTH(_last), 1)')):
        measures[name] = (f"VAR _last = [Window end]\nVAR _first = {start_expression}\nRETURN CALCULATE([Active users], FILTER(ALL('Date'), 'Date'[date] >= _first && 'Date'[date] <= _last))", '#,##0')
    measures['MAU-30 coverage'] = ('VAR _last = [Window end]\nVAR _days = CALCULATE([Covered days], FILTER(ALL(\'Date\'), \'Date\'[date] >= _last - 29 && \'Date\'[date] <= _last))\nRETURN FORMAT(_days, "0") & " / 30 days; " & IF(_days = 30, "complete", "partial")', '')
    measures['Calendar month coverage'] = ("VAR _last = [Window end]\nVAR _first = DATE(YEAR(_last), MONTH(_last), 1)\nRETURN CALCULATE([Usage coverage], FILTER(ALL('Date'), 'Date'[date] >= _first && 'Date'[date] <= _last))", '')
    return measures

def model_definition(workspace_id, lakehouse_id, table_prefix='inv_'):
    uuid.UUID(workspace_id)
    uuid.UUID(lakehouse_id)
    if not re.fullmatch('[A-Za-z][A-Za-z0-9_]*', table_prefix):
        raise ValueError('The inventory table prefix is not a safe identifier.')
    parts = {}
    parts['definition.pbism'] = json.dumps({'version': '4.2', 'settings': {'qnaEnabled': False}}, indent=2)
    parts['definition/database.tmdl'] = 'database\n\tcompatibilityLevel: 1702\n\tcompatibilityMode: powerBI\n'
    parts['definition/model.tmdl'] = 'model Model\n\tculture: en-US\n\tdefaultPowerBIDataSourceVersion: powerBI_V3\n\tdiscourageImplicitMeasures\n\tdirectLakeBehavior: directLakeOnly\n\n' + '\n'.join(('ref table ' + tmdl_name(alias) for alias in ALIASES.values())) + '\n'
    parts['definition/expressions.tmdl'] = f'expression InventoryOneLake =\n\t\tlet\n\t\t\tSource = AzureStorage.DataLake("https://onelake.dfs.fabric.microsoft.com/{workspace_id}/{lakehouse_id}", [HierarchicalNavigation=true])\n\t\tin\n\t\t\tSource\n'
    measures = formula_measures()
    hidden = {'row_id', 'tenant_id', 'run_id', 'schema_version', 'scope_fingerprint', 'capacity_key', 'workspace_key', 'item_key'}
    for key, alias in ALIASES.items():
        lines = [f'table {tmdl_name(alias)}']
        if key == 'calendar':
            lines.append('\tdataCategory: Time')
        if key == 'published_runs':
            for name, (expression, fmt) in measures.items():
                lines.extend(['', f'\tmeasure {tmdl_name(name)} = ```', *['\t\t\t' + line for line in expression.splitlines()], '\t\t\t```', '\t\tformatString: ' + (fmt or '""')])
        for name, kind in {**COMMON, **TABLES[key]}.items():
            lines.extend(['', f'\tcolumn {tmdl_name(name)}', '\t\tdataType: ' + {'long': 'int64', 'date': 'dateTime'}.get(kind, kind), '\t\tsummarizeBy: none', '\t\tsourceColumn: ' + name])
            if name in hidden and (not (key == 'published_runs' and name == 'run_id')):
                lines.append('\t\tisHidden')
            if name == 'url':
                lines.append('\t\tdataCategory: WebUrl')
            if kind == 'date':
                lines.append('\t\tformatString: yyyy-MM-dd')
            if key == 'calendar' and name == 'month_name':
                lines.append('\t\tsortByColumn: month_number')
        lines.extend(['', f'\tpartition {tmdl_name(alias)} = entity', '\t\tmode: directLake', '\t\tsource', '\t\t\tentityName: ' + table_prefix + key, '\t\t\tschemaName: dbo', '\t\t\texpressionSource: InventoryOneLake', ''])
        parts[f'definition/tables/{key}.tmdl'] = '\n'.join(lines)
    relationships = [('workspaces', 'capacity_key', 'capacities', 'capacity_key'), ('items', 'workspace_key', 'workspaces', 'workspace_key'), ('access_assignments', 'workspace_key', 'workspaces', 'workspace_key'), ('report_view_events', 'event_date', 'calendar', 'date_key')]
    for key in ('reports', 'semantic_models', 'tables', 'columns', 'measures', 'relationships', 'data_sources', 'lineage', 'endpoints', 'shortcuts', 'refresh_status', 'findings'):
        relationships.append((key, 'item_key', 'items', 'item_key'))
    parts['definition/relationships.tmdl'] = '\n\n'.join((f"relationship '{object_name(*relation)}'\n\tfromColumn: {tmdl_name(ALIASES[relation[0]])}.{tmdl_name(relation[1])}\n\ttoColumn: {tmdl_name(ALIASES[relation[2]])}.{tmdl_name(relation[3])}\n\tcrossFilteringBehavior: oneDirection" for relation in relationships)) + '\n'
    return parts

def field(table, name, measure=False, source=None):
    ref = {'Source': source} if source else {'Entity': table}
    return {'Measure' if measure else 'Column': {'Expression': {'SourceRef': ref}, 'Property': name}}

def projection(table, name, measure=False):
    return {'field': field(table, name, measure), 'queryRef': table + '.' + name, 'displayName': name.replace('_', ' ')}

def positive_measure_filter(name, seed):
    return {'name': object_name(seed, name), 'field': field('Runs', name, True), 'type': 'Advanced', 'filter': {'Version': 2, 'From': [{'Name': 'm', 'Entity': 'Runs', 'Type': 0}], 'Where': [{'Condition': {'Comparison': {'ComparisonKind': 2, 'Left': field('Runs', name, True, 'm'), 'Right': {'Literal': {'Value': '0L'}}}}}]}, 'isHiddenInViewMode': True}

def visual(seed, kind, query, x, y, width, height, title, row_filter=None):
    data = {'$schema': SCHEMA_ROOT + 'definition/visualContainer/1.0.0/schema.json', 'name': object_name(seed, title, str(x), str(y)), 'position': {'x': x, 'y': y, 'z': 0, 'width': width, 'height': height, 'tabOrder': y * 10 + x}, 'visual': {'visualType': kind, 'query': {'queryState': query}, 'visualContainerObjects': {'title': [{'properties': {'show': {'expr': {'Literal': {'Value': 'true'}}}, 'text': {'expr': {'Literal': {'Value': "'" + title.replace("'", "''") + "'"}}}}}]}}}
    if row_filter:
        data['filterConfig'] = {'filters': [positive_measure_filter(row_filter, seed + title)]}
    return data

def report_definition(model_id):
    uuid.UUID(model_id)
    parts = {'definition.pbir': {'$schema': SCHEMA_ROOT + 'definitionProperties/2.0.0/schema.json', 'version': '4.0', 'datasetReference': {'byConnection': {'connectionString': 'semanticmodelid=' + model_id}}}, 'definition/version.json': {'$schema': SCHEMA_ROOT + 'definition/versionMetadata/1.0.0/schema.json', 'version': '2.0.0'}, 'definition/report.json': {'$schema': SCHEMA_ROOT + 'definition/report/1.0.0/schema.json', 'themeCollection': {'baseTheme': {'name': 'CY24SU06', 'reportVersionAtImport': '5.55', 'type': 'SharedResources'}}, 'layoutOptimization': 'None'}}
    pages = []

    def add_page(label, drill=None):
        page_id = object_name('page', label)
        pages.append(page_id)
        page = {'$schema': SCHEMA_ROOT + 'definition/page/1.0.0/schema.json', 'name': page_id, 'displayName': label, 'displayOption': 'FitToPage', 'height': 900, 'width': 1440}
        if drill:
            table, column = drill
            filter_id = object_name('drill', label)
            page['visibility'] = 'HiddenInViewMode'
            page['filterConfig'] = {'filters': [{'name': filter_id, 'field': field(table, column), 'type': 'Categorical', 'howCreated': 'Drillthrough'}]}
            page['pageBinding'] = {'name': object_name('binding', label), 'type': 'Drillthrough', 'acceptsFilterContext': 'Default', 'parameters': [{'name': object_name('parameter', label), 'boundFilter': filter_id, 'fieldExpr': field(table, column)}]}
        parts[f'definition/pages/{page_id}/page.json'] = page
        return page_id

    def add(page_id, value):
        parts[f"definition/pages/{page_id}/visuals/{value['name']}/visual.json"] = value

    def slicer(page, table, name, x, title, y=12):
        add(page, visual(page, 'slicer', {'Values': {'projections': [projection(table, name)]}}, x, y, 340, 110, title))

    def card(page, measure, x, y, width=330):
        add(page, visual(page, 'card', {'Values': {'projections': [projection('Runs', measure, True)]}}, x, y, width, 105, measure))

    def table(page, source, columns, x, y, width, height, title, row_filter=None, measures=None):
        projections = [projection(source, column) for column in columns]
        projections += [projection('Runs', measure, True) for measure in measures or []]
        if row_filter and row_filter not in (measures or []):
            projections.append(projection('Runs', row_filter, True))
        add(page, visual(page, 'tableEx', {'Values': {'projections': projections}}, x, y, width, height, title, row_filter))

    def standard_controls(page):
        slicer(page, 'Runs', 'run_id', 16, 'Snapshot: clear selection for latest')
        slicer(page, 'Capacities', 'capacity_name', 370, 'Capacity')
        slicer(page, 'Workspaces', 'workspace_name', 724, 'Workspace')
        card(page, 'Snapshot status', 1090, 12)
    overview = add_page('Overview')
    standard_controls(overview)
    for x, metric in zip((16, 370, 724, 1090), ('Rows Capacities', 'Rows Workspaces', 'Rows Items', 'Collection gaps')):
        card(overview, metric, x, 145)
    add(overview, visual(overview, 'clusteredBarChart', {'Category': {'projections': [projection('Items', 'item_type')]}, 'Y': {'projections': [projection('Runs', 'Rows Items', True)]}}, 16, 275, 690, 580, 'Observed item types', 'Rows Items'))
    table(overview, 'Collection status', ['collector', 'source_scope', 'status', 'row_count'], 730, 275, 690, 580, 'Coverage and unavailable collections', 'Rows Collection status')
    inventory = add_page('Inventory')
    standard_controls(inventory)
    table(inventory, 'Workspaces', ['workspace_name', 'workspace_id', 'workspace_type', 'domain_id', 'state'], 16, 145, 690, 310, 'Workspaces: right-click for detail', 'Rows Workspaces')
    table(inventory, 'Capacities', ['capacity_name', 'sku', 'region', 'state'], 730, 145, 690, 310, 'Capacities, including empty ones', 'Rows Capacities')
    table(inventory, 'Items', ['item_name', 'item_type', 'workspace_id', 'item_id', 'owner_name', 'url'], 16, 480, 1404, 375, 'All observed items: right-click for detail', 'Rows Items')
    usage = add_page('Report usage')
    standard_controls(usage)
    add(usage, visual(usage, 'slicer', {'Values': {'projections': [projection('Date', 'date')]}}, 16, 145, 340, 105, 'Observation date range'))
    for x, metric in zip((370, 724, 1090), ('Report views', 'Active users', 'Usage coverage')):
        card(usage, metric, x, 145)
    for x, metric in zip((16, 300, 584, 868, 1152), ('WAU', 'MAU-30', 'MAU-30 coverage', 'Calendar month active users', 'Calendar month coverage')):
        card(usage, metric, x, 275, 268)
    add(usage, visual(usage, 'lineChart', {'Category': {'projections': [projection('Date', 'date')]}, 'Y': {'projections': [projection('Runs', 'Report views', True)]}}, 16, 405, 690, 440, 'Views by observed UTC day'))
    table(usage, 'Views', ['report_name', 'workspace_name', 'item_id'], 730, 405, 690, 440, 'Observed report activity', 'Report views', ['Report views', 'Active users', 'Last observed view'])
    access = add_page('Access')
    standard_controls(access)
    slicer(access, 'Access', 'principal_name', 16, 'Principal', y=145)
    table(access, 'Access', ['entity_name', 'entity_type', 'principal_name', 'principal_type', 'permission', 'assignment_scope', 'inheritance'], 16, 280, 1404, 305, 'Observed assignments, not complete effective data permissions', 'Rows Access')
    table(access, 'Group members', ['group_id', 'member_name', 'member_type', 'membership_kind'], 16, 610, 690, 245, 'Available group expansion', 'Rows Group members')
    table(access, 'Collection status', ['collector', 'entity_id', 'status', 'message'], 730, 610, 690, 245, 'Unresolved sources and groups', 'Rows Collection status')
    changes = add_page('Changes and attention')
    standard_controls(changes)
    table(changes, 'Changes', ['entity_type', 'entity_id', 'change_type', 'field_name', 'before_value', 'after_value'], 16, 145, 1404, 440, 'Changes versus the previous comparable snapshot', 'Rows Changes')
    table(changes, 'Findings', ['finding_type', 'workspace_id', 'item_id', 'summary', 'evidence'], 16, 610, 1404, 245, 'Review candidates: no automatic remediation', 'Rows Findings')
    workspace = add_page('Workspace detail', ('Workspaces', 'workspace_id'))
    standard_controls(workspace)
    table(workspace, 'Workspaces', ['workspace_name', 'description', 'capacity_id', 'domain_id', 'url'], 16, 145, 1404, 240, 'Selected workspace', 'Rows Workspaces')
    table(workspace, 'Items', ['item_name', 'item_type', 'item_id', 'url'], 16, 410, 690, 440, 'Workspace items', 'Rows Items')
    table(workspace, 'Access', ['principal_name', 'permission', 'assignment_scope'], 730, 410, 690, 440, 'Recorded workspace and item assignments', 'Rows Access')
    item = add_page('Item detail', ('Items', 'item_id'))
    standard_controls(item)
    table(item, 'Items', ['item_name', 'item_type', 'description', 'creator_name', 'owner_name', 'url'], 16, 145, 1404, 220, 'Selected item', 'Rows Items')
    table(item, 'Tables', ['table_name', 'schema_name', 'parent_type', 'table_type'], 16, 390, 690, 210, 'Available table metadata', 'Rows Tables')
    table(item, 'Model measures', ['table_name', 'measure_name', 'data_type', 'description'], 730, 390, 690, 210, 'Available measure metadata', 'Rows Model measures')
    table(item, 'Lineage', ['source_item_id', 'target_item_id', 'relationship_type'], 16, 625, 1404, 220, 'Known dependency edges', 'Rows Lineage')
    parts['definition/pages/pages.json'] = {'$schema': SCHEMA_ROOT + 'definition/pagesMetadata/1.0.0/schema.json', 'pageOrder': pages, 'activePageName': overview}
    return parts

def encode_definition(parts, format_name=None):
    result = {'parts': [{'path': name, 'payloadType': 'InlineBase64', 'payload': base64.b64encode((value if isinstance(value, str) else json.dumps(value, ensure_ascii=True)).encode()).decode()} for name, value in parts.items()]}
    if format_name:
        result['format'] = format_name
    return result

def write_definitions(root, parts):
    root = Path(root)
    for name, value in parts.items():
        destination = root.joinpath(*name.split('/'))
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_text((value if isinstance(value, str) else json.dumps(value, indent=2, ensure_ascii=True)) + '\n', encoding='utf-8')


In [ ]:
"""Notebook-only onboarding and optional report creation.

Definitions have no side effects. Only an explicit action creates resources.
"""


class NotebookSetupError(ValueError):
    def __init__(self, code, problem, action):
        self.code = code
        self.problem = problem
        self.action = action
        super().__init__(f"[{code}] {problem}\nNext step: {action}")


def setup_help():
    print(
        "SETUP NEEDED / HELP\n"
        "1. Upload this notebook to a Fabric workspace on an active Fabric capacity.\n"
        "2. Attach a dedicated, schema-enabled inventory lakehouse as the default.\n"
        "   If none exists, set MODE='setup_lakehouse' and choose SETUP_LAKEHOUSE_NAME.\n"
        "   Then attach the returned lakehouse in the notebook Explorer and restart the session.\n"
        "3. For the first inventory, set MODE='collect_and_view' and CONFIRM_WRITE_TARGET=True.\n"
        "4. For normal use, leave MODE='view_saved'. Browsing does not collect or write data.\n"
        "5. Optional: set MODE='create_report' after a snapshot exists. No local scripts are needed.\n"
        "Keep WRITE_MODE='append_snapshot' and CONFIRM_OVERWRITE=False.\n"
        "No workspace, capacity, permissions, credentials, or schedule is created automatically."
    )


def notebook_tenant_id():
    import base64
    import notebookutils

    token = notebookutils.credentials.getToken("pbi")
    segment = token.split(".")[1]
    claims = json.loads(base64.urlsafe_b64decode(segment + "=" * (-len(segment) % 4)))
    tenant = identifier(claims.get("tid"))
    try:
        uuid.UUID(tenant)
    except ValueError:
        raise NotebookSetupError(
            "IDENTITY_CONTEXT", "The Fabric session did not provide a tenant identifier.",
            "Sign in to the intended Fabric tenant. Never paste an access token into this notebook.",
        ) from None
    return tenant


def notebook_context():
    try:
        import notebookutils
    except ImportError:
        raise NotebookSetupError(
            "FABRIC_RUNTIME_REQUIRED", "This action requires a Fabric PySpark notebook.",
            "Upload the notebook to Fabric. Local Jupyter does not provide its runtime identity or lakehouse context.",
        ) from None
    context = notebookutils.runtime.context
    if not context.get("currentWorkspaceId"):
        raise NotebookSetupError(
            "WORKSPACE_CONTEXT", "The notebook has no Fabric workspace context.",
            "Save it in a Fabric workspace on an active capacity, then start a session.",
        )
    return context


def resolve_inventory_settings(settings, context, tenant_provider=notebook_tenant_id):
    mode = settings["mode"]
    if mode not in {"help", "setup_lakehouse", "view_saved", "collect_and_view", "create_report"}:
        raise NotebookSetupError(
            "MODE_INVALID", f"Unknown MODE: {mode!r}.",
            "Use help, setup_lakehouse, view_saved, collect_and_view, or create_report.",
        )
    if mode in {"help", "setup_lakehouse"}:
        raise NotebookSetupError("MODE_CONTEXT", "This mode does not read inventory tables.", "Use the action entry point.")
    mapping = {
        "destination_workspace_id": "defaultLakehouseWorkspaceId",
        "destination_workspace_name": "defaultLakehouseWorkspaceName",
        "destination_lakehouse_id": "defaultLakehouseId",
        "destination_lakehouse_name": "defaultLakehouseName",
    }
    config = dict(settings)
    for field, context_key in mapping.items():
        actual = context.get(context_key)
        if not actual:
            raise NotebookSetupError(
                "DEFAULT_LAKEHOUSE_REQUIRED", "No complete default-lakehouse binding is available.",
                "Attach a dedicated, schema-enabled inventory lakehouse in Explorer and restart the session. "
                "Use MODE='setup_lakehouse' if you need to create one first.",
            )
        explicit = settings.get(field)
        compare = identifier if field.endswith("_id") else str
        if explicit and compare(explicit) != compare(actual):
            raise NotebookSetupError(
                "DESTINATION_MISMATCH", f"{field} does not match the attached lakehouse context.",
                "Check the displayed workspace and lakehouse. Clear an obsolete advanced override or attach the intended target. "
                "No destination was changed.",
            )
        config[field] = actual
    config["tenant_id"] = identifier(settings.get("tenant_id") or tenant_provider())
    config["action_mode"] = mode
    config["mode"] = "collect_and_view" if mode == "collect_and_view" else "view_saved"
    validate_config(config)
    if mode == "collect_and_view" and settings.get("confirm_write_target") is not True:
        raise NotebookSetupError(
            "WRITE_TARGET_CONFIRMATION",
            f"Inventory would be written to {config['destination_workspace_name']} / {config['destination_lakehouse_name']}.",
            "Confirm that this is a dedicated inventory lakehouse, then set CONFIRM_WRITE_TARGET=True. "
            "This does not authorize overwrites; CONFIRM_OVERWRITE remains separate.",
        )
    return config


def provisioning_error(operation, error, created=None):
    reason = safe_text(str(error), 500)
    text = (str(error.code) + " " + reason).lower()
    if error.status == 429:
        next_step = "Wait for the service retry window, then rerun this action. Completed managed resources will be reused."
    elif "scope" in text:
        next_step = (
            "The session token lacks a required API scope. A higher workspace role alone will not fix that. "
            "Use an approved Fabric authentication context or the Fabric UI; ask your identity administrator if consent is needed."
        )
    elif "licen" in text:
        next_step = (
            "Confirm that the report author has an appropriate Power BI Pro, PPU, or applicable trial license. "
            "A Fabric capacity does not replace the author's license requirement."
        )
    elif "capacity" in text:
        next_step = (
            "Ask the capacity administrator to confirm an assigned, active Fabric capacity. "
            "This notebook never resumes, resizes, or purchases capacity."
        )
    elif error.status == 401:
        next_step = "Check the signed-in account and tenant. Renew the Fabric session; do not paste credentials into the notebook."
    elif error.status == 403:
        next_step = (
            "Ask the workspace administrator to confirm Contributor or higher authoring rights in this destination. "
            "For reports, also check the Power BI license and source-lakehouse read access. "
            "Tenant Fabric administrator rights alone do not grant workspace authoring access."
        )
    elif error.status == 404:
        next_step = "Check the destination and item IDs. A 404 can also hide a resource you cannot access; do not infer deletion."
    else:
        next_step = "Review the service code and destination. Correct the cause before rerunning; no automatic rollback is performed."
    progress = ""
    if created:
        progress = " Managed resources already created: " + json.dumps(created, sort_keys=True) + "."
    return NotebookSetupError(
        "PROVISIONING_FAILED",
        f"{operation} failed (HTTP {error.status}; {error.code}): {reason}.{progress} Inventory tables were not overwritten.",
        next_step,
    )


def wait_notebook_operation(api, response, result_needed=False):
    if response.status_code != 202:
        return response.json() if response.content else {}
    location = response.headers.get("Location")
    if not location:
        raise NotebookSetupError(
            "OPERATION_LOCATION_MISSING", "The service accepted a request but returned no operation location.",
            "Inspect the destination before retrying so that you do not create a duplicate.",
        )
    deadline = time.monotonic() + 900
    while time.monotonic() < deadline:
        header = response.headers.get("Retry-After", "15")
        time.sleep(max(5, int(header) if header.isdigit() else 15))
        response = api.request("GET", location)
        status = response.json()
        if status.get("status") == "Succeeded":
            return api.get(location.rstrip("/") + "/result") if result_needed else status
        if status.get("status") in {"Failed", "Cancelled"}:
            raise NotebookSetupError(
                "OPERATION_FAILED", "The Fabric operation did not succeed: " + safe_text(json.dumps(status.get("error", {})), 700),
                "Review this operation in Fabric. Retry only after checking whether an item was created.",
            )
    raise NotebookSetupError(
        "OPERATION_PENDING", "The operation did not finish within the monitoring window.",
        "Monitor the existing operation before retrying: " + api.url(location),
    )


def require_usable_workspace(api, workspace_id):
    workspace = api.get(f"/v1/workspaces/{workspace_id}")
    if not workspace.get("capacityId"):
        raise NotebookSetupError(
            "CAPACITY_REQUIRED", "The destination workspace has no Fabric capacity assignment.",
            "Ask the workspace/capacity administrator to assign an active Fabric capacity. "
            "Power BI Pro or PPU alone does not provide Spark compute.",
        )
    try:
        capacities = list(api.pages("/v1/capacities"))
    except ApiFailure as error:
        if error.status not in (401, 403):
            raise
        print("Capacity listing is not permitted for this identity. Workspace operations do not require capacity-administrator rights.")
        return workspace
    capacity = next((c for c in capacities if identifier(c["id"]) == identifier(workspace["capacityId"])), None)
    if capacity:
        if str(capacity.get("state", "")).lower() != "active":
            raise NotebookSetupError(
                "CAPACITY_INACTIVE", "The destination capacity is not active.",
                "Ask its administrator to resume it if appropriate; resuming can incur charges. The notebook does not do this.",
            )
        if str(capacity.get("sku", "")).upper().startswith("PP"):
            raise NotebookSetupError(
                "FABRIC_CAPACITY_REQUIRED", "A PPU allocation does not provide Fabric Spark capacity.",
                "Use a workspace assigned to an active Fabric capacity.",
            )
    else:
        print("Capacity state is not visible through your capacity permissions. The service will enforce availability.")
    return workspace


def setup_inventory_lakehouse(settings, context, fabric_api):
    name = settings.get("setup_lakehouse_name", "")
    if not re.fullmatch(r"[A-Za-z][A-Za-z0-9_]*", name):
        raise NotebookSetupError(
            "LAKEHOUSE_NAME_INVALID", "SETUP_LAKEHOUSE_NAME must use letters, digits, and underscores and start with a letter.",
            "Choose a dedicated name, for example FabricInventory. Do not use a business-data lakehouse.",
        )
    workspace_id = identifier(context["currentWorkspaceId"])
    try:
        require_usable_workspace(fabric_api, workspace_id)
        items = list(fabric_api.pages(f"/v1/workspaces/{workspace_id}/items"))
        matches = [i for i in items if i["type"] == "Lakehouse" and i["displayName"] == name]
        if len(matches) > 1:
            raise NotebookSetupError("NAME_COLLISION", "More than one lakehouse has this name.", "Choose a unique setup name.")
        if matches:
            lakehouse = matches[0]
            created = False
        else:
            response = fabric_api.request("POST", f"/v1/workspaces/{workspace_id}/items", json={
                "displayName": name, "type": "Lakehouse", "creationPayload": {"enableSchemas": True},
            })
            lakehouse = wait_notebook_operation(fabric_api, response, True)
            created = True
    except ApiFailure as error:
        raise provisioning_error("Lakehouse setup", error) from None
    return {
        "status": "attach_lakehouse", "created": created, "workspace_id": workspace_id,
        "lakehouse_id": lakehouse["id"], "lakehouse_name": lakehouse["displayName"],
        "next_step": "Attach this dedicated lakehouse as the default in Explorer, restart the session, "
                     "then select collect_and_view and confirm the write target. No notebook definition was overwritten.",
    }


def create_notebook_report(settings, config, store, fabric_api, powerbi_api):
    if store.select_run() is None:
        raise NotebookSetupError(
            "SNAPSHOT_REQUIRED", "No published inventory is available for the report.",
            "Run collect_and_view first. You can still use notebook views without a Power BI report.",
        )
    workspace_id = config["destination_workspace_id"]
    marker = f"Fabric Inventory Explorer; lakehouse={config['destination_lakehouse_id']}; prefix={config['table_prefix']}"
    model_name = settings.get("report_model_name", "Fabric Inventory Model")
    report_name = settings.get("report_name", "Fabric Inventory")
    if not model_name.strip() or not report_name.strip():
        raise NotebookSetupError("REPORT_NAME_REQUIRED", "Report/model names must not be blank.", "Set REPORT_NAME and REPORT_MODEL_NAME.")
    created = {}
    try:
        require_usable_workspace(fabric_api, workspace_id)
        items = list(fabric_api.pages(f"/v1/workspaces/{workspace_id}/items"))

        def resolve(kind, name, explicit_id):
            if explicit_id:
                item = next((i for i in items if identifier(i["id"]) == identifier(explicit_id)), None)
                if item is None or item["type"] != kind:
                    raise NotebookSetupError(
                        "REPORT_ID_MISMATCH", f"The supplied {kind} ID is not visible in the inventory workspace.",
                        "Check the ID and your access. No item was replaced.",
                    )
                return item
            matching = [i for i in items if i["type"] == kind and i["displayName"] == name]
            if len(matching) > 1:
                raise NotebookSetupError("NAME_COLLISION", f"Multiple {kind} items match {name!r}.", "Choose a unique name.")
            if matching and matching[0].get("description") != marker:
                raise NotebookSetupError(
                    "NAME_COLLISION", f"{name!r} already exists and is not identified as this inventory's managed item.",
                    "Choose a different report/model name, or supply its ID only after confirming it belongs to this inventory. "
                    "Existing definitions are never overwritten by create_report.",
                )
            return matching[0] if matching else None

        model = resolve("SemanticModel", model_name, settings.get("report_model_id"))
        report = resolve("Report", report_name, settings.get("report_id"))
        if report and not model:
            raise NotebookSetupError("REPORT_MODEL_REQUIRED", "An existing report was selected without its model.",
                                     "Supply the matching REPORT_MODEL_ID. No resource was changed.")
        if model and report:
            actual = powerbi_api.get(f"/v1.0/myorg/groups/{workspace_id}/reports/{report['id']}")
            if identifier(actual.get("datasetId")) != identifier(model["id"]):
                raise NotebookSetupError("REPORT_BINDING_MISMATCH", "The selected report uses another semantic model.",
                                         "Select the matching IDs or choose new names. No report was rebound.")
        else:
            if not model:
                parts = model_definition(workspace_id, config["destination_lakehouse_id"], config["table_prefix"])
                response = fabric_api.request("POST", f"/v1/workspaces/{workspace_id}/semanticModels", json={
                    "displayName": model_name, "description": marker, "definition": encode_definition(parts, "TMDL"),
                })
                model = wait_notebook_operation(fabric_api, response, True)
                created["model_id"] = model["id"]
            response = powerbi_api.request("POST", f"/v1.0/myorg/groups/{workspace_id}/datasets/{model['id']}/refreshes",
                                          json={"type": "Full", "commitMode": "transactional", "applyRefreshPolicy": False})
            location = response.headers.get("Location")
            if not location:
                raise NotebookSetupError("REFRESH_PENDING", "Model refresh was accepted without a location.",
                                         "Check refresh history before retrying. Model ID: " + model["id"])
            deadline = time.monotonic() + 900
            while time.monotonic() < deadline:
                time.sleep(10)
                result = powerbi_api.get(location)
                if result.get("status") == "Completed":
                    break
                if result.get("status") in {"Failed", "Cancelled", "Disabled"}:
                    raise NotebookSetupError(
                        "MODEL_REFRESH_FAILED", "The model could not read the inventory tables: " + safe_text(json.dumps(result), 600),
                        "Confirm the model owner can read this lakehouse and all inventory tables. A workspace role does not replace source access.",
                    )
            else:
                raise NotebookSetupError("REFRESH_PENDING", "The existing model refresh is still pending.",
                                         "Monitor the refresh before retrying. Model ID: " + model["id"])
            parts = report_definition(model["id"])
            response = fabric_api.request("POST", f"/v1/workspaces/{workspace_id}/reports", json={
                "displayName": report_name, "description": marker, "definition": encode_definition(parts, "PBIR"),
            })
            report = wait_notebook_operation(fabric_api, response, True)
            created["report_id"] = report["id"]
    except ApiFailure as error:
        raise provisioning_error("Power BI report setup", error, created) from None
    return {
        "status": "report_ready", "created": created,
        "model_id": model["id"], "report_id": report["id"],
        "report_url": f"https://app.powerbi.com/groups/{workspace_id}/reports/{report['id']}?ctid={config['tenant_id']}",
        "note": "No sharing permissions, fixed identity, source data, or existing definitions were changed. "
                "Default SSO still requires viewers to have suitable inventory-data access.",
    }


def initialize_notebook(settings, spark_session, graph_credential=None):
    mode = settings["mode"]
    if mode == "help":
        setup_help()
        return {"status": "help", "config": None, "store": None}
    if mode not in {"setup_lakehouse", "view_saved", "collect_and_view", "create_report"}:
        raise NotebookSetupError("MODE_INVALID", f"Unknown MODE: {mode!r}.", "Use one of the five documented modes.")
    context = notebook_context()
    if mode == "setup_lakehouse":
        tenant = identifier(settings.get("tenant_id") or notebook_tenant_id())
        fabric_api, _, _, _ = make_notebook_apis({"tenant_id": tenant})
        result = setup_inventory_lakehouse(settings, context, fabric_api)
        print(json.dumps(result, indent=2))
        return {**result, "config": None, "store": None}
    if mode == "view_saved" and not context.get("defaultLakehouseId"):
        setup_help()
        print("No default lakehouse is attached. No inventory was read, collected, or written.")
        return {"status": "needs_setup", "config": None, "store": None}
    config = resolve_inventory_settings(settings, context)
    assert_notebook_destination(config)
    print("Resolved destination:", config["destination_workspace_name"], "/", config["destination_lakehouse_name"])
    print("Tenant:", config["tenant_id"], "| Action:", mode)
    store = SparkInventoryStore(spark_session, config)
    result = {"status": "ready", "config": config, "store": store}
    if mode == "collect_and_view":
        try:
            publication = run_collection(store, config, graph_credential)
        except ApiFailure as error:
            if error.status == 401:
                action = "Check the signed-in tenant and renew the Fabric session. A persistent authentication failure is not a permission fallback."
            else:
                scope = "admin" if "/admin/" in error.url else "user"
                action = collection_guidance("inventory", failure_kind(error), scope, str(error))
            raise NotebookSetupError(
                "COLLECTION_STOPPED",
                f"Collection stopped (HTTP {error.status}; {error.code}): {safe_text(str(error), 500)}. "
                "No new snapshot was published; previous published snapshots remain available.",
                action,
            ) from None
        result["run_id"] = publication["run_id"]
        print("Published inventory:", publication["run_id"], publication["status"])
    elif mode == "create_report":
        fabric_api, powerbi_api, _, _ = make_notebook_apis(config)
        result.update(create_notebook_report(settings, config, store, fabric_api, powerbi_api))
        print(json.dumps({k: v for k, v in result.items() if k not in {"config", "store"}}, indent=2))
    else:
        print("Read-only browsing. No tenant collection, provisioning, or data writes were requested.")
    return result


## Safe setup and optional collection

Only `collect_and_view` enters the inventory collection and Delta-write path.
`setup_lakehouse` and `create_report` use separate, explicit resource-creation paths.
The destination check prevents writing to a different attached lakehouse.
The writer lock rejects concurrent collection. It never removes another run's lock.
Expected source-access gaps are recorded. Unexpected errors prevent publication.

The explorer shows only published snapshots. Valid archive events become visible through published per-day coverage.
Failed or partial extraction days are not presented as complete usage.

In [ ]:
"""Embedded action entry point; help and viewing do not provision resources."""

USER_SETTINGS = {
    "mode": MODE,
    "tenant_id": TENANT_ID,
    "destination_workspace_id": INVENTORY_WORKSPACE_ID,
    "destination_workspace_name": INVENTORY_WORKSPACE_NAME,
    "destination_lakehouse_id": INVENTORY_LAKEHOUSE_ID,
    "destination_lakehouse_name": INVENTORY_LAKEHOUSE_NAME,
    "confirm_write_target": CONFIRM_WRITE_TARGET,
    "setup_lakehouse_name": SETUP_LAKEHOUSE_NAME,
    "report_model_name": REPORT_MODEL_NAME,
    "report_name": REPORT_NAME,
    "report_model_id": REPORT_MODEL_ID,
    "report_id": REPORT_ID,
    "capacity_ids": COLLECT_CAPACITY_IDS,
    "workspace_ids": COLLECT_WORKSPACE_IDS,
    "item_types": COLLECT_ITEM_TYPES,
    "include_personal": INCLUDE_PERSONAL_WORKSPACES,
    "include_deep": INCLUDE_DEEP_METADATA,
    "include_permissions": INCLUDE_PERMISSIONS,
    "include_usage": INCLUDE_USAGE,
    "expand_groups": EXPAND_GROUPS,
    "activity_days": ACTIVITY_BACKFILL_DAYS,
    "write_mode": WRITE_MODE,
    "confirm_overwrite": CONFIRM_OVERWRITE,
    "table_prefix": TABLE_PREFIX,
}
BOOTSTRAP = initialize_notebook(USER_SETTINGS, globals().get("spark"), GRAPH_CREDENTIAL)
CONFIG = BOOTSTRAP.get("config")
store = BOOTSTRAP.get("store")
if BOOTSTRAP.get("run_id"):
    VIEW["run_id"] = BOOTSTRAP["run_id"]


## Explorer controls

Choose a view, set filters, then select **Apply view**.
Copy entity IDs from the tables to move from a capacity to a workspace, item, or principal.
Blank snapshot selects the latest published run. A selected run remains fixed until you apply new controls.
The single output area clears previous results when filters change.

Use **Collection browser** to reach every allowlisted saved table and all fields.
Each table shows matching row counts and stable pages. Charts show bounded summaries, never the only copy of a result.
If widgets are unavailable, edit `VIEW` and rerun the cell below.

Filters apply where the selected collection has the relevant scope or field.
Use Access for principal/permission filters and Usage for UTC date filters.

In [ ]:
explorer = None
if store is not None:
    explorer = InventoryExplorer(store, VIEW, display)
    explorer.controls()
else:
    print("Current setup status:", BOOTSTRAP["status"])
    print("Read the recipient guide before selecting the next action.")


## What each view provides

| View | Use |
|---|---|
| Overview | Observed totals, item-type chart, and collection coverage |
| Capacities | Every returned capacity, SKU/state/region, and child counts |
| Workspaces | Workspaces by capacity, domains, descriptions, and item counts |
| Items | Every returned type, recorded owners/creators, labels, dates, and links |
| Deep metadata | Models, tables, columns, measures, relationships, shortcuts, and endpoints |
| Access | Reported grants, transitive group-derived paths, and unresolved groups |
| Usage | Views, distinct users, weekly/monthly windows, rankings, and missing observation days |
| Dependencies | Known upstream/downstream edges with source evidence |
| Changes | Comparable snapshot differences and evidence-backed attention items |
| Diagnostics | Collector errors, unsupported or paused sources, published and unfinished runs |
| Collection browser | Table index, full schemas, paged records, and exact field inspection |

**Usage definitions:** views are deduplicated `ViewReport` events, excluding explicitly failed events.
Human MAU counts distinct logged actors identified as regular users or administrators.
Unknown and automated actors are not silently counted as humans.
Calendar-month figures end at the selected UTC date and are month-to-date when that date is inside the month.
Thirty-day figures remain partial until the archive covers all 30 days.

**Access definitions:** paths are assignments and recorded group memberships.
They do not evaluate SQL grants, RLS/OLS, or all OneLake restrictions.
Missing group expansion means unresolved membership, not zero members.

## Direct view recipes

Use these methods when you prefer ordinary cells over widgets.
Each method reads saved inventory only.

```python
explorer.update({**VIEW, "view": "Workspaces", "capacity_id": "<capacity ID>"})
explorer.render()

explorer.update({**VIEW, "view": "Items", "workspace_id": "<workspace ID>"})
explorer.render()

explorer.update({**VIEW, "view": "Deep metadata", "item_id": "<item ID>", "collection": "columns"})
explorer.render()

explorer.update({**VIEW, "view": "Access", "principal_id": "<principal ID>"})
explorer.render()

explorer.update({**VIEW, "view": "Changes", "run_id": "<new run>", "baseline_run_id": "<old run>"})
explorer.render()

explorer.record("items", item_id="<item ID>")
full_frame = explorer.frame("items")
```

`full_frame` contains the complete filtered result, not only the displayed page.
Use `row_id` for an unambiguous all-column record detail.

## Recovery and privacy

- A failed run is not published as current inventory.
- Diagnostics lists unfinished runs. Inspect the failed cell and its Fabric job record.
- If a stale `Files/inventory_control/writer.lock` remains, confirm the earlier writer stopped before removing that exact file.
- Do not remove source files, delete prior snapshots, or bypass the overwrite variable to recover a failed run.
- Reruns append a new snapshot. Event IDs prevent duplicate archived views.
- Source capacities, permissions, and tenant settings are not changed by this notebook.
- Keep this workspace private. Saved outputs can contain administrative identifiers.
- Tokens, passwords, client IPs, full expressions, and raw HTTP responses are not stored.
- Manual runs do not create schedules. Long gaps can exceed activity-log retention.

The Power BI report reads the same saved facts and metric definitions.
The notebook remains usable before the report is deployed.

## Documentation graph: why these boundaries exist

The design was cross-checked against the local Microsoft documentation vault's Fabric link graph.
The vault itself is not uploaded or required at runtime. These links open the official source articles.

| Documentation path | Consequence for this notebook |
|---|---|
| [Notebook visualization](https://learn.microsoft.com/en-us/fabric/data-engineering/notebook-visualization) -> [Notebook exploration](https://learn.microsoft.com/en-us/fabric/data-engineering/lakehouse-notebook-explore) | Native display defaults to 1,000 rows and supports at most 10,000. Explicit pages keep all inventory records reachable. |
| [Metadata scanning](https://learn.microsoft.com/en-us/fabric/governance/metadata-scanning-overview) -> [Scanner setup](https://learn.microsoft.com/en-us/fabric/admin/metadata-scanning-setup) | Schema details depend on tenant settings. Missing details are labeled; the notebook never enables settings. |
| [Permission model](https://learn.microsoft.com/en-us/fabric/security/permission-model) -> [OneLake access model](https://learn.microsoft.com/en-us/fabric/onelake/security/data-access-control-model) | Workspace/item grants and data-level authorization are different layers. Access paths are evidence, not an effective-access verdict. |
| [Semantic Link](https://learn.microsoft.com/en-us/fabric/data-science/semantic-link-overview) -> [Direct Lake](https://learn.microsoft.com/en-us/fabric/fundamentals/direct-lake-overview) | Collection uses Semantic Link clients; the companion report reads the saved Delta inventory. |
| [Direct Lake development](https://learn.microsoft.com/en-us/fabric/fundamentals/direct-lake-develop) -> [Direct Lake security](https://learn.microsoft.com/en-us/fabric/fundamentals/direct-lake-security-integration) | Model owner and viewer access must be checked. A copied inventory does not inherit every source security rule. |

The documentation graph describes documentation links. It is not evidence of dependencies between your tenant's Fabric items.

In [ ]:
"""Embedded notebook validation and optional job exit value."""

if VALIDATE_ALL_VIEWS:
    if MODE != "view_saved":
        raise ValueError("Run view validation separately in view_saved mode.")
    if explorer is None or explorer.run is None:
        raise ValueError("View validation requires a published snapshot.")
    for view_name in InventoryExplorer.VIEWS:
        explorer.render(view_name)

    from functools import reduce
    from pyspark.sql import functions as F

    captured_pages = []
    pager = InventoryExplorer(
        store, {**VIEW, "page_size": 100},
        lambda frame: captured_pages.append(frame.select("row_id")),
    )
    all_items = pager.frame("items")
    expected_rows = all_items.count()
    for number in range(1, (expected_rows + 99) // 100 + 1):
        pager.page(all_items, "Paging check", page=number)
    if captured_pages:
        combined = reduce(lambda left, right: left.unionByName(right), captured_pages)
        assert combined.count() == expected_rows
        assert combined.distinct().count() == expected_rows

    filtered = InventoryExplorer(
        store, {**VIEW, "workspace_id": INVENTORY_WORKSPACE_ID}, lambda frame: None,
    )
    expected_filtered = store.snapshot("items", explorer.run["run_id"]).where(
        F.col("workspace_id") == INVENTORY_WORKSPACE_ID,
    ).count()
    assert filtered.frame("items").count() == expected_filtered
    older = store.published().where(
        (F.col("run_id") != explorer.run["run_id"])
        & (F.col("scope_fingerprint") == explorer.run["scope_fingerprint"])
    ).orderBy(F.desc("published_at_utc")).limit(1).collect()
    if older:
        comparison = InventoryExplorer(
            store, {**VIEW, "baseline_run_id": older[0].run_id}, display,
        )
        comparison.changes()

    try:
        validate_config({
            **CONFIG, "mode": "collect_and_view",
            "write_mode": "replace_inventory", "confirm_overwrite": False,
        })
    except PermissionError:
        print("Unconfirmed replacement is rejected before any write.")
    else:
        raise AssertionError("Overwrite protection failed.")
    print("Live paging, workspace filtering, history, and overwrite guard checks completed.")

if JOB_EXIT:
    import notebookutils

    saved = store.select_run(VIEW.get("run_id") or None) if store is not None else None
    result = {
        "mode": MODE,
        "run_id": saved["run_id"] if saved else None,
        "status": (saved["status"] if saved else "no_published_inventory")
                  if MODE in {"view_saved", "collect_and_view"} else BOOTSTRAP["status"],
        "inventory_status": saved["status"] if saved else None,
        "workspace_count": saved["workspace_count"] if saved else None,
        "item_count": saved["item_count"] if saved else None,
        "coverage_gap_count": saved["coverage_gap_count"] if saved else None,
        "views_checked": VALIDATE_ALL_VIEWS,
    }
    for key in ("created", "model_id", "report_id", "report_url", "lakehouse_id", "lakehouse_name", "next_step"):
        if key in BOOTSTRAP:
            result[key] = BOOTSTRAP[key]
    notebookutils.notebook.exit(json.dumps(result))
